In [1]:
import datetime as dt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import requests
import re
import pickle 
import pickle as pk
import keras
import tensorflow as tf
import os

from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest

from torchdiffeq import odeint

from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
from sklearn.metrics import confusion_matrix
import plotly.graph_objs as go
import matplotlib.dates as mdates
from plotly.subplots import make_subplots
import plotly.express as px

import matplotlib.pyplot as plt
import seaborn as sns


# Pickle Save object
def save_obj(obj, filename):
    filename = 'CSV data/' + filename + '.pkl'
    with open(filename, 'wb') as outp:  
        pickle.dump(obj, outp, pickle.HIGHEST_PROTOCOL)

# Pickle Read Object
def read_obj(filename):
    
    filename = 'CSV data/' + filename + '.pkl'
    with open(filename, 'rb') as inp: 
        return pickle.load(inp)

In [2]:
### Visualization 2020 to 2023

df = pd.read_csv("./CSV data/GT/GT_final.csv")
df.head()

start_date = '2020-01-01 00:00:00'
num_rows = len(df)
date_range = pd.date_range(start=start_date, periods=num_rows, freq='H')

df['date'] = date_range
df.to_csv("./CSV data/GT/GT_final_date.csv", index = False)

In [14]:
df = pd.read_csv("./CSV data/GT/GT_final_date.csv")

In [4]:
### spike 처리 

date = df['date']
df = df[df.iloc[:, :-1] < 300]
df = df[df.iloc[:, :-1] > -300]
df['date'] = date

In [3]:
def detect_and_visualize_building_anomalies(model, data_df, window_size=50, 
                                          date_column=0, n_features_to_show=5, 
                                          threshold_method='adaptive_f1',
                                          use_plotly=True):
    
    # 1. 데이터 준비
    if isinstance(data_df, pd.DataFrame):
        dates = pd.to_datetime(data_df.iloc[:, date_column])
        sensor_data = data_df.iloc[:, 1:].values.astype(float)
    else:
        # numpy array인 경우
        dates = pd.to_datetime(data_df[:, date_column])
        sensor_data = data_df[:, 1:].astype(float)
    
    feature_names = [f"Sensor_{i+1}" for i in range(sensor_data.shape[1])]
    
    print(f"데이터 형태: {sensor_data.shape}")
    print(f"날짜 범위: {dates.min()} ~ {dates.max()}")
    print(f"센서 개수: {len(feature_names)}")
    
    # 2. 윈도우 분할 및 이상 점수 계산
    print(f"\n윈도우 크기 {window_size}로 분할 중...")
    
    device = next(model.parameters()).device
    model.eval()
    
    # 윈도우 분할
    windows = []
    window_dates = []
    
    for i in range(0, len(sensor_data) - window_size + 1, 1):
        windows.append(sensor_data[i:i+window_size])
        window_dates.append(dates.iloc[i + window_size//2])  # 윈도우 중간점
    
    windows = np.array(windows)
    print(f"생성된 윈도우 개수: {len(windows)}")
    
    # 배치별 이상 점수 계산
    batch_size = 64
    all_scores = []
    
    with torch.no_grad():
        for start_idx in range(0, len(windows), batch_size):
            end_idx = min(start_idx + batch_size, len(windows))
            batch_data = windows[start_idx:end_idx]
            
            batch_tensor = torch.tensor(batch_data, dtype=torch.float32).to(device)
            T = batch_tensor.shape[1]
            timestamps = torch.linspace(0, 1, T).unsqueeze(0).expand(batch_tensor.shape[0], -1).to(device)
            
            try:
                scores = model.anomaly_score(batch_tensor, timestamps)
                if isinstance(scores, torch.Tensor):
                    scores = scores.cpu().numpy()
                all_scores.extend(scores)
            except Exception as e:
                print(f"배치 {start_idx//batch_size} 오류: {e}")
                all_scores.extend([0.5] * len(batch_data))
    
    all_scores = np.array(all_scores)
    
    # 점수 정규화
    all_scores = (all_scores - np.min(all_scores)) / (np.max(all_scores) - np.min(all_scores) + 1e-8)
    
    # 3. 임계값 계산 (pseudo 라벨 사용)
    pseudo_labels = (all_scores > np.percentile(all_scores, 95)).astype(int)  # 상위 5%를 이상치로 가정
    
    try:
        threshold = improved_threshold_selection(all_scores, pseudo_labels, threshold_method)
    except:
        threshold = np.percentile(all_scores, 90)  # 기본 임계값
    
    anomaly_mask = all_scores > threshold
    
    print(f"\n이상치 감지 결과:")
    print(f"임계값: {threshold:.4f}")
    print(f"이상치 윈도우 개수: {np.sum(anomaly_mask)}")
    print(f"이상치 비율: {np.mean(anomaly_mask):.2%}")
    
    # 4. 시각화
    if use_plotly:
        visualize_with_plotly(dates, sensor_data, window_dates, all_scores, 
                            threshold, anomaly_mask, feature_names, n_features_to_show)
    else:
        visualize_with_matplotlib(dates, sensor_data, window_dates, all_scores, 
                                threshold, anomaly_mask, feature_names, n_features_to_show)
    
    # 5. 이상치 구간 정보 반환
    anomaly_periods = []
    if np.any(anomaly_mask):
        anomaly_dates = np.array(window_dates)[anomaly_mask]
        
        print(f"\n🚨 감지된 이상치 구간:")
        for i, date in enumerate(anomaly_dates[:10]):  # 상위 10개만 표시
            score = all_scores[anomaly_mask][i]
            print(f"  {date.strftime('%Y-%m-%d %H:%M')} (점수: {score:.4f})")
            anomaly_periods.append({'date': date, 'score': score})
    
    return {
        'anomaly_scores': all_scores,
        'threshold': threshold,
        'anomaly_mask': anomaly_mask,
        'window_dates': window_dates,
        'anomaly_periods': anomaly_periods
    }

def visualize_with_plotly(dates, sensor_data, window_dates, scores, threshold, anomaly_mask, 
                         feature_names, n_features_to_show):
    """Plotly를 사용한 인터랙티브 시각화"""
    
    n_features = min(n_features_to_show, len(feature_names))
    
    # 서브플롯 생성 (센서 데이터 + 이상 점수)
    fig = make_subplots(
        rows=n_features + 1, 
        cols=1,
        shared_xaxes=True,
        subplot_titles=feature_names[:n_features] + ['Anomaly Score'],
        vertical_spacing=0.03
    )
    
    # 센서 데이터 플롯
    for i in range(n_features):
        # 원본 데이터
        fig.add_trace(
            go.Scatter(
                x=dates, 
                y=sensor_data[:, i],
                mode='lines',
                name=feature_names[i],
                line=dict(color='blue', width=1),
                showlegend=True if i == 0 else False
            ),
            row=i+1, col=1
        )
        
        # 이상치 구간 강조
        anomaly_dates = np.array(window_dates)[anomaly_mask]
        if len(anomaly_dates) > 0:
            for anomaly_date in anomaly_dates:
                fig.add_vline(
                    x=anomaly_date,
                    line=dict(color="red", width=2, dash="dash"),
                    opacity=0.7,
                    row=i+1, col=1
                )
    
    # 이상 점수 플롯
    fig.add_trace(
        go.Scatter(
            x=window_dates,
            y=scores,
            mode='lines',
            name='Anomaly Score',
            line=dict(color='orange', width=2)
        ),
        row=n_features+1, col=1
    )
    
    # 임계값 선
    fig.add_hline(
        y=threshold,
        line=dict(color="red", width=2, dash="dot"),
        annotation_text=f"Threshold: {threshold:.3f}",
        row=n_features+1, col=1
    )
    
    # 이상치 포인트 강조
    anomaly_scores = scores[anomaly_mask]
    anomaly_dates_scores = np.array(window_dates)[anomaly_mask]
    
    if len(anomaly_scores) > 0:
        fig.add_trace(
            go.Scatter(
                x=anomaly_dates_scores,
                y=anomaly_scores,
                mode='markers',
                name='Anomalies',
                marker=dict(color='red', size=8, symbol='x')
            ),
            row=n_features+1, col=1
        )
    
    # 레이아웃 설정
    fig.update_layout(
        title="🏢 빌딩 데이터 이상치 감지 결과",
        height=150 * (n_features + 1),
        showlegend=True,
        hovermode='x unified'
    )
    
    fig.update_xaxes(title_text="Date", row=n_features+1, col=1)
    
    fig.show()

def visualize_with_matplotlib(dates, sensor_data, window_dates, scores, threshold, anomaly_mask,
                            feature_names, n_features_to_show):
    """Matplotlib을 사용한 기본 시각화"""
    
    n_features = min(n_features_to_show, len(feature_names))
    
    fig, axes = plt.subplots(n_features + 1, 1, figsize=(15, 3 * (n_features + 1)), sharex=True)
    
    if n_features == 0:
        axes = [axes]
    
    # 센서 데이터 플롯
    for i in range(n_features):
        ax = axes[i] if n_features > 1 else axes[0]
        
        # 원본 데이터
        ax.plot(dates, sensor_data[:, i], 'b-', linewidth=1, alpha=0.7, label=feature_names[i])
        
        # 이상치 구간 강조
        anomaly_dates = np.array(window_dates)[anomaly_mask]
        for anomaly_date in anomaly_dates:
            ax.axvline(x=anomaly_date, color='red', linestyle='--', alpha=0.7)
        
        ax.set_ylabel(feature_names[i])
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # 이상 점수 플롯
    score_ax = axes[-1]
    score_ax.plot(window_dates, scores, 'orange', linewidth=2, label='Anomaly Score')
    score_ax.axhline(y=threshold, color='red', linestyle=':', linewidth=2, label=f'Threshold: {threshold:.3f}')
    
    # 이상치 포인트 강조
    anomaly_scores = scores[anomaly_mask]
    anomaly_dates_scores = np.array(window_dates)[anomaly_mask]
    
    if len(anomaly_scores) > 0:
        score_ax.scatter(anomaly_dates_scores, anomaly_scores, color='red', s=50, marker='x', 
                        label='Anomalies', zorder=5)
    
    score_ax.set_ylabel('Anomaly Score')
    score_ax.set_xlabel('Date')
    score_ax.legend()
    score_ax.grid(True, alpha=0.3)
    
    # 날짜 형식 설정
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.suptitle('🏢 빌딩 데이터 이상치 감지 결과', fontsize=16, y=0.98)
    plt.show()

# 사용 예시:
"""
# CSV 파일에서 데이터 로드
data_df = pd.read_csv('building_data.csv')

# 이상치 감지 및 시각화
results = detect_and_visualize_building_anomalies(
    model=model,
    data_df=data_df,
    window_size=50,
    date_column=0,  # 첫 번째 열이 날짜
    n_features_to_show=5,  # 상위 5개 센서만 표시
    threshold_method='adaptive_f1',
    use_plotly=True  # 인터랙티브 시각화
)
"""

"\n# CSV 파일에서 데이터 로드\ndata_df = pd.read_csv('building_data.csv')\n\n# 이상치 감지 및 시각화\nresults = detect_and_visualize_building_anomalies(\n    model=model,\n    data_df=data_df,\n    window_size=50,\n    date_column=0,  # 첫 번째 열이 날짜\n    n_features_to_show=5,  # 상위 5개 센서만 표시\n    threshold_method='adaptive_f1',\n    use_plotly=True  # 인터랙티브 시각화\n)\n"

### Neural ODE

In [4]:
def evaluate_model_sections(model, test_data_list, test_labels_list, device, batch_size=32):

    model.eval()
    
    # Fix device consistency
    model_device = next(model.parameters()).device
    
    # Ensure all model submodules are on the same device
    for module in model.modules():
        module.to(model_device)
    
    all_metrics = {}
    
    for section_idx, (test_data, test_labels) in enumerate(zip(test_data_list, test_labels_list)):
        print(f"Section {section_idx} evaluation...")
        
        # Store original shapes for debugging
        original_data_shape = test_data.shape if hasattr(test_data, 'shape') else np.array(test_data).shape
        original_labels_shape = test_labels.shape if hasattr(test_labels, 'shape') else np.array(test_labels).shape
        
        # Convert to tensors
        if not isinstance(test_data, torch.Tensor):
            test_data = torch.FloatTensor(test_data)
        if not isinstance(test_labels, torch.Tensor):
            test_labels = torch.FloatTensor(test_labels)
            
        # Move to device
        test_data = test_data.to(model_device)
        test_labels = test_labels.to(model_device)
        
        # Handle data shape - should be 3D (batch, seq_len, features)
        if len(test_data.shape) == 2:
            # Single sequence: (seq_len, features) -> (1, seq_len, features)
            test_data = test_data.unsqueeze(0)
            # Single label for the sequence
            if len(test_labels.shape) == 1:
                test_labels = torch.tensor([test_labels.max().item()], device=model_device)
        elif len(test_data.shape) == 3:
            # Already windowed data: (num_windows, window_size, features)
            # Labels should match: (num_windows,)
            if len(test_labels.shape) == 1 and test_labels.shape[0] == test_data.shape[0]:
                # Correct format - keep as is
                pass
            else:
                raise ValueError(f"Label shape {test_labels.shape} doesn't match data batch size {test_data.shape[0]}")
        else:
            raise ValueError(f"Invalid data shape. Expected 2D or 3D, got: {test_data.shape}")
        
        print(f"Data shape: {original_data_shape} -> {test_data.shape}")
        print(f"Labels shape: {original_labels_shape} -> {test_labels.shape}")
        
        # Create time steps
        seq_len = test_data.shape[1]
        time_steps = torch.linspace(0, 1, seq_len).to(model_device)
        
        # Process in batches
        all_scores = []
        num_samples = test_data.shape[0]
        
        with torch.no_grad():
            for i in range(0, num_samples, batch_size):
                batch_data = test_data[i:i+batch_size]
                batch_size_actual = batch_data.shape[0]
                
                # Expand time steps for batch
                batch_time = time_steps.unsqueeze(0).repeat(batch_size_actual, 1)
                
                # Calculate anomaly scores
                scores = model.anomaly_score(batch_data, batch_time)
                all_scores.append(scores.cpu().numpy())
        
        # Combine all scores
        anomaly_scores = np.concatenate(all_scores, axis=0)
        true_labels = test_labels.cpu().numpy()
        
        # Set threshold (95th percentile)
        threshold = np.percentile(anomaly_scores, 95)
        predictions = (anomaly_scores > threshold).astype(int)
        
        try:
            auc = roc_auc_score(true_labels, anomaly_scores)
        except:
            auc = 0.0
            
        precision = precision_score(true_labels, predictions, zero_division=0)
        recall = recall_score(true_labels, predictions, zero_division=0)
        f1 = f1_score(true_labels, predictions, zero_division=0)
        accuracy = accuracy_score(true_labels, predictions)
        
        # Store results
        metrics = {
            'threshold': threshold,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'auc': auc,
            'accuracy': accuracy,
            'anomaly_scores': anomaly_scores,
            'predictions': predictions,
            'true_labels': true_labels
        }
        
        all_metrics[f'section_{section_idx}'] = metrics
        
        # Print results
        print(f"   Threshold: {threshold:.4f}")
        print(f"   Precision: {precision:.4f}")
        print(f"   Recall: {recall:.4f}")
        print(f"   F1 Score: {f1:.4f}")
        print(f"   AUC: {auc:.4f}")
        print(f"   Accuracy: {accuracy:.4f}")
    
    # Calculate overall averages
    avg_metrics = {}
    for metric in ['precision', 'recall', 'f1_score', 'auc', 'accuracy']:
        avg_metrics[f'avg_{metric}'] = np.mean([metrics[metric] for metrics in all_metrics.values()])
    
    all_metrics['average'] = avg_metrics
    
    print(f"\nOverall Performance:")
    print(f"   Avg Precision: {avg_metrics['avg_precision']:.4f}")
    print(f"   Avg Recall: {avg_metrics['avg_recall']:.4f}")
    print(f"   Avg F1 Score: {avg_metrics['avg_f1_score']:.4f}")
    print(f"   Avg AUC: {avg_metrics['avg_auc']:.4f}")
    print(f"   Avg Accuracy: {avg_metrics['avg_accuracy']:.4f}")
    
    return all_metrics


def visualize_anomaly_detection_fixed(model, test_data, test_labels, threshold=None, sample_idx=0, figsize=(15, 8)):
    """
    이상 탐지 결과를 시각화하는 함수
    
    Args:
        model: 훈련된 모델
        test_data: 테스트 데이터
        test_labels: 실제 라벨
        threshold: 이상 탐지 임계값 (None이면 자동 계산)
        sample_idx: 시각화할 샘플 인덱스
        figsize: 그래프 크기
    """
    model.eval()
    
    # 데이터를 torch 텐서로 변환
    if not isinstance(test_data, torch.Tensor):
        device = next(model.parameters()).device
        test_data = torch.FloatTensor(test_data).to(device)
        test_labels = torch.FloatTensor(test_labels).to(device)
    
    # 시간 스텝 생성
    seq_len = test_data.shape[1]
    time_steps = torch.linspace(0, 1, seq_len).to(test_data.device)
    
    # 이상 점수 계산
    with torch.no_grad():
        time_batch = time_steps.unsqueeze(0).repeat(test_data.shape[0], 1)
        anomaly_scores = model.anomaly_score(test_data, time_batch)
        
        # 재구성 데이터 계산
        encoded = model.encoder(test_data, time_batch)
        z0 = encoded[:, -1, :]
        reconstructed = model.decoder(z0, time_batch)
    
    # numpy로 변환
    test_data_np = test_data.cpu().numpy()
    test_labels_np = test_labels.cpu().numpy()
    anomaly_scores_np = anomaly_scores.cpu().numpy()
    reconstructed_np = reconstructed.cpu().numpy()
    
    # 임계값 설정
    if threshold is None:
        threshold = np.percentile(anomaly_scores_np, 95)
    
    # 예측 생성
    predictions = (anomaly_scores_np > threshold).astype(int)
    
    # 시각화
    fig, axes = plt.subplots(4, 1, figsize=figsize)
    
    # 시간 축 생성
    time_axis = np.arange(seq_len)
    
    # 1. 원본 데이터와 재구성 데이터 비교 (첫 번째 특성)
    axes[0].plot(time_axis, test_data_np[sample_idx, :, 0], 'b-', label='원본 데이터', linewidth=2)
    axes[0].plot(time_axis, reconstructed_np[sample_idx, :, 0], 'r--', label='재구성 데이터', linewidth=2)
    axes[0].set_title(f'샘플 {sample_idx}: 원본 vs 재구성 데이터 (첫 번째 특성)', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2. 이상 점수
    axes[1].plot(time_axis, [anomaly_scores_np[sample_idx]] * seq_len, 'g-', linewidth=3, label=f'이상 점수: {anomaly_scores_np[sample_idx]:.4f}')
    axes[1].axhline(y=threshold, color='r', linestyle='--', linewidth=2, label=f'임계값: {threshold:.4f}')
    axes[1].set_title('이상 점수 vs 임계값', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # 3. 실제 라벨
    axes[2].plot(time_axis, [test_labels_np[sample_idx]] * seq_len, 'b-', linewidth=3, label=f'실제 라벨: {"이상" if test_labels_np[sample_idx] == 1 else "정상"}')
    axes[2].set_title('실제 라벨', fontsize=12, fontweight='bold')
    axes[2].set_ylim(-0.1, 1.1)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    # 4. 예측 결과
    prediction_color = 'r' if predictions[sample_idx] == 1 else 'g'
    axes[3].plot(time_axis, [predictions[sample_idx]] * seq_len, prediction_color + '-', linewidth=3, 
                label=f'예측: {"이상" if predictions[sample_idx] == 1 else "정상"}')
    axes[3].set_title('예측 결과', fontsize=12, fontweight='bold')
    axes[3].set_ylim(-0.1, 1.1)
    axes[3].legend()
    axes[3].grid(True, alpha=0.3)
    axes[3].set_xlabel('시간 스텝', fontsize=10)
    
    # 전체 제목
    result_text = "정확" if test_labels_np[sample_idx] == predictions[sample_idx] else "오류"
    fig.suptitle(f'이상 탐지 시각화 - 샘플 {sample_idx} ({result_text})', 
                fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # 통계 정보 출력
    print(f"\n📊 샘플 {sample_idx} 상세 정보:")
    print(f"   🎯 실제 라벨: {'이상' if test_labels_np[sample_idx] == 1 else '정상'}")
    print(f"   🔍 예측 결과: {'이상' if predictions[sample_idx] == 1 else '정상'}")
    print(f"   📈 이상 점수: {anomaly_scores_np[sample_idx]:.6f}")
    print(f"   📏 임계값: {threshold:.6f}")
    print(f"   ✅ 판정: {'정확' if test_labels_np[sample_idx] == predictions[sample_idx] else '오류'}")
    
    # 재구성 오차 계산
    recon_error = np.mean(np.square(test_data_np[sample_idx] - reconstructed_np[sample_idx]))
    print(f"   🔧 재구성 오차: {recon_error:.6f}")

print("evaluate_model_sections, visualize_anomaly_detection_fixed")


evaluate_model_sections, visualize_anomaly_detection_fixed


In [5]:
def train_enhanced_anomaly_detector(model, train_loader, optimizer, num_epochs, device):
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for x_batch, time_steps in train_loader:
            x_batch = x_batch.to(device)
            time_steps = time_steps.to(device)
            
            # 1. 정상 데이터 처리
            encoded_normal = model.encoder(x_batch, time_steps)
            z0_normal = encoded_normal[:, -1, :]
            reconstructed_normal = model.decoder(z0_normal, time_steps)
            
            # 재구성 손실
            recon_loss = F.mse_loss(reconstructed_normal, x_batch)
            
            # 2. 인공 이상치 생성
            noise_level = min(0. + epoch * 0.01, 0.5)  # 점진적 노이즈 증가
            noise_mask = (torch.rand_like(x_batch[:, :, 0:1]) < 0.2).float()
            noise = torch.randn_like(x_batch) * noise_level * noise_mask
            noisy_x = x_batch + noise
            
            # 이상치 데이터 처리
            encoded_anomaly = model.encoder(noisy_x, time_steps)
            z0_anomaly = encoded_anomaly[:, -1, :]
            reconstructed_anomaly = model.decoder(z0_anomaly, time_steps)
            
            # 3. 특징 대조 손실
            normal_features = encoded_normal.mean(dim=1)
            anomaly_features = encoded_anomaly.mean(dim=1)
            contrast_loss = feature_contrastive_loss(normal_features, anomaly_features, margin=2.0)
            
            # 4. 시간적 일관성 손실
            normal_scores = model(x_batch, time_steps)
            anomaly_scores = model(noisy_x, time_steps)
            temporal_loss = temporal_consistency_loss(normal_scores) + temporal_consistency_loss(anomaly_scores)
            
            # 5. 분류 손실
            class_loss = F.binary_cross_entropy(normal_scores, torch.zeros_like(normal_scores)) + \
                         F.binary_cross_entropy(anomaly_scores, torch.ones_like(anomaly_scores) * 0.9)
            
            # 최종 손실 계산
            loss = 0.4 * recon_loss + 0.3 * contrast_loss + 0.1 * temporal_loss + 0.2 * class_loss
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # 그래디언트 클리핑
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.6f}")
        

In [6]:
# 통합된 이상 탐지 평가 함수
def evaluate_anomaly_detector(
    model, 
    test_data=None, 
    test_labels=None, 
    test_loader=None,
    device=None, 
    threshold=None,
    threshold_method='auto',
    threshold_percentile=95,
    return_scores=False,
    verbose=True
):

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = model.to(device)
    model.eval()
    
    # 입력 검증
    if test_loader is not None and (test_data is not None or test_labels is not None):
        raise ValueError("test_loader와 test_data/test_labels를 동시에 사용할 수 없습니다.")
    
    if test_loader is None and (test_data is None or test_labels is None):
        raise ValueError("test_loader 또는 test_data/test_labels를 제공해야 합니다.")
    
    # 이상 점수 계산
    anomaly_scores = []
    all_labels = []
    
    with torch.no_grad():
        if test_loader is not None:
            # DataLoader 사용 경우
            for batch_data in test_loader:
                if len(batch_data) == 3:  # (x, times, mask)
                    x_batch, times_batch, mask_batch = batch_data
                    x_batch = x_batch.to(device)
                    times_batch = times_batch.to(device)
                    mask_batch = mask_batch.to(device)
                    
                    scores = model.anomaly_score(x_batch, times_batch, mask_batch)
                    anomaly_scores.extend(scores.cpu().numpy())
                else:
                    raise ValueError("DataLoader의 배치 형태가 예상과 다릅니다.")
            
            # test_loader에서 라벨 추출 (별도로 제공되어야 함)
            if test_labels is None:
                raise ValueError("test_loader 사용시 test_labels를 별도로 제공해야 합니다.")
            all_labels = test_labels
            
        else:
            # 직접 데이터 입력 경우
            if isinstance(test_data, np.ndarray):
                test_data = torch.tensor(test_data, dtype=torch.float32)
            
            # 차원 처리
            if test_data.dim() == 2:  # (T, D) → (1, T, D)
                test_data = test_data.unsqueeze(0)
            
            # 시간 벡터 생성
            T = test_data.shape[1]
            timestamps = torch.linspace(0, 1, T).unsqueeze(0).expand(test_data.shape[0], -1).to(device)
            
            # 마스크 생성
            mask = torch.ones_like(timestamps)
            
            # 데이터를 디바이스로 이동
            test_data = test_data.to(device)
            
            # 이상 점수 계산
            scores = model.anomaly_score(test_data, timestamps, mask)
            anomaly_scores = scores.cpu().numpy()
            all_labels = test_labels
    
    # 점수와 라벨 형태 맞추기
    anomaly_scores = np.array(anomaly_scores)
    all_labels = np.array(all_labels)
    
    # 차원 조정
    if all_labels.ndim == 1 and len(anomaly_scores.shape) > 1 and anomaly_scores.shape[0] == 1:
        # 단일 시퀀스 (1, T)와 벡터 라벨 (T,)
        scores_flat = anomaly_scores.squeeze()
        labels_flat = all_labels
    elif all_labels.ndim == 2:
        # 다중 시퀀스와 2D 라벨
        scores_flat = anomaly_scores.reshape(-1)
        labels_flat = all_labels.reshape(-1)
    elif len(anomaly_scores.shape) > 1:
        # 다중 시퀀스와 1D 라벨 - 윈도우별 최대값 사용
        scores_flat = np.max(anomaly_scores, axis=1) if anomaly_scores.ndim > 1 else anomaly_scores
        labels_flat = all_labels
    else:
        scores_flat = anomaly_scores
        labels_flat = all_labels
    
    # 임계값 설정
    if threshold is None:
        if threshold_method == 'percentile':
            threshold = np.percentile(scores_flat, threshold_percentile)
        elif threshold_method == 'normal_based':
            # 정상 데이터 기반 임계값
            normal_scores = scores_flat[labels_flat == 0]
            if len(normal_scores) > 0:
                threshold = np.mean(normal_scores) + 2 * np.std(normal_scores)
            else:
                threshold = np.percentile(scores_flat, 95)
        elif threshold_method == 'auto':
            # 자동 선택: 정상 데이터가 충분하면 normal_based, 아니면 percentile
            normal_scores = scores_flat[labels_flat == 0]
            if len(normal_scores) > len(scores_flat) * 0.7:  # 정상 데이터가 70% 이상
                threshold = np.mean(normal_scores) + 2 * np.std(normal_scores)
            else:
                threshold = np.percentile(scores_flat, threshold_percentile)
        else:
            raise ValueError(f"지원하지 않는 threshold_method: {threshold_method}")
    
    # 예측 생성
    predictions = (scores_flat > threshold).astype(int)
    
    # 성능 지표 계산
    metrics = {}
    
    try:
        # 기본 분류 지표
        metrics['accuracy'] = accuracy_score(labels_flat, predictions)
        metrics['precision'] = precision_score(labels_flat, predictions, average='binary', zero_division=0)
        metrics['recall'] = recall_score(labels_flat, predictions, average='binary', zero_division=0)
        metrics['f1'] = f1_score(labels_flat, predictions, average='binary', zero_division=0)
        
        # ROC-AUC (레이블이 다양한 경우에만)
        if len(np.unique(labels_flat)) > 1:
            metrics['auc'] = roc_auc_score(labels_flat, scores_flat)
        else:
            metrics['auc'] = None
            
    except Exception as e:
        if verbose:
            print(f"지표 계산 중 오류 발생: {e}")
        metrics['accuracy'] = 0.0
        metrics['precision'] = 0.0
        metrics['recall'] = 0.0
        metrics['f1'] = 0.0
        metrics['auc'] = None
    
    # 추가 정보
    metrics['threshold'] = threshold
    metrics['threshold_method'] = threshold_method
    metrics['num_samples'] = len(labels_flat)
    metrics['num_anomalies'] = np.sum(labels_flat)
    metrics['num_predicted_anomalies'] = np.sum(predictions)
    
    # 이상 점수 반환 옵션
    if return_scores:
        metrics['anomaly_scores'] = scores_flat
        metrics['predictions'] = predictions
        metrics['labels'] = labels_flat
    
    # 결과 출력
    if verbose:
        print(f"=== 이상 탐지 평가 결과 ===")
        print(f"Threshold ({threshold_method}): {threshold:.4f}")
        print(f"Accuracy: {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall: {metrics['recall']:.4f}")
        print(f"F1 Score: {metrics['f1']:.4f}")
        if metrics['auc'] is not None:
            print(f"AUC: {metrics['auc']:.4f}")
        print(f"Samples: {metrics['num_samples']}, "
              f"True Anomalies: {metrics['num_anomalies']}, "
              f"Predicted: {metrics['num_predicted_anomalies']}")
    
    return metrics

In [7]:
def plot_spike_detection_for_variable(x, time_steps, anomaly_scores, var_idx=0, threshold=None, sample_idx=0, percentile=95):
    """
    x: (batch, seq_len, input_dim) - 입력 시계열
    time_steps: (batch, seq_len) - 시간 정보
    anomaly_scores: (batch, seq_len) - 모델 출력 anomaly score
    var_idx: int - 시각화할 변수 인덱스 (ex. 0~127)
    sample_idx: int - 배치 중 어떤 시계열을 볼지 (기본값: 0)
    threshold: float - anomaly 판단 기준값. 없으면 percentile 기반 계산
    percentile: float - threshold를 anomaly_scores의 상위 몇 %로 지정할지
    """

    # Tensor → NumPy
    x_np = x.detach().cpu().numpy()
    time_np = time_steps.detach().cpu().numpy()
    score_np = anomaly_scores.detach().cpu().numpy()

    # 하나의 시계열 추출
    values = x_np[sample_idx, :, var_idx]
    times = time_np[sample_idx]
    scores = score_np[sample_idx]

    # threshold 계산 (percentile 기반 or 수동 입력)
    if threshold is None:
        threshold = np.percentile(scores, percentile)
    
    # spike 위치
    spikes = scores > threshold

    # 시각화
    fig, ax1 = plt.subplots(figsize=(14, 5))

    # 실제 값 plot
    ax1.set_title(f"Spike Detection for Variable {var_idx}")
    ax1.plot(times, values, label=f'Variable {var_idx} Value', color='blue', marker='o')
    ax1.set_xlabel("Time")
    ax1.set_ylabel("Variable Value", color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')

    # spike 점 찍기
    ax1.scatter(times[spikes], values[spikes], color='red', label='Detected Spikes', zorder=5)

    # anomaly score plot (우측 축)
    ax2 = ax1.twinx()
    ax2.plot(times, scores, color='orange', linestyle='--', label='Anomaly Score')
    ax2.axhline(threshold, color='gray', linestyle=':', label='Threshold')
    ax2.set_ylabel("Anomaly Score", color='orange')
    ax2.tick_params(axis='y', labelcolor='orange')

    # 범례 정리
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # 간단 출력
    print("Total spikes detected:", spikes.sum())
    print("Threshold used:", threshold)


In [8]:
def train_irregular_ode(model, data_dict, num_epochs=30):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    # Optimizer setup
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=True
    )
    
    # Training
    model.train()
    history = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        epoch_recon_loss = 0
        epoch_kl_loss = 0
        epoch_flow_loss = 0
        
        for batch_idx, (x_batch, times_batch, mask_batch) in enumerate(data_dict['train_loader']):
            x_batch = x_batch.to(device)
            times_batch = times_batch.to(device)
            mask_batch = mask_batch.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(x_batch, times_batch, mask_batch)
            losses = outputs["losses"]
            
            # Backward pass
            losses["total_loss"].backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            # Accumulate losses
            epoch_loss += losses["total_loss"].item()
            epoch_recon_loss += losses["recon_loss"].item()
            epoch_kl_loss += losses["kl_loss"].item()
            epoch_flow_loss += losses["flow_loss"].item()
        
        # Average losses per epoch
        n_batches = len(data_dict['train_loader'])
        avg_loss = epoch_loss / n_batches
        avg_recon_loss = epoch_recon_loss / n_batches
        avg_kl_loss = epoch_kl_loss / n_batches
        avg_flow_loss = epoch_flow_loss / n_batches
        
        # Learning rate adjustment
        scheduler.step(avg_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs}, "
              f"Loss: {avg_loss:.4f}, "
              f"Recon: {avg_recon_loss:.4f}, "
              f"KL: {avg_kl_loss:.4f}, "
              f"Flow: {avg_flow_loss:.4f}")
        
        history.append({
            'epoch': epoch + 1,
            'total_loss': avg_loss,
            'recon_loss': avg_recon_loss,
            'kl_loss': avg_kl_loss,
            'flow_loss': avg_flow_loss
        })
    
    # Evaluation
    model.eval()
    anomaly_scores = []
    
    with torch.no_grad():
        for x_batch, times_batch, mask_batch in data_dict['test_loader']:
            x_batch = x_batch.to(device)
            times_batch = times_batch.to(device)
            mask_batch = mask_batch.to(device)
            
            # Calculate anomaly scores
            scores = model.anomaly_score(x_batch, times_batch, mask_batch)
            anomaly_scores.extend(scores)
    
    anomaly_scores = np.array(anomaly_scores)
    
    # Convert labels to numpy and check dimensions
    if isinstance(data_dict['lab_tst'], torch.Tensor):
        test_labels = data_dict['lab_tst'].cpu().numpy()
    else:
        test_labels = np.array(data_dict['lab_tst'])
    
    # Adjust dimensions
    if test_labels.ndim > 1:
        test_labels = test_labels.flatten()
    if anomaly_scores.ndim > 1:
        anomaly_scores = anomaly_scores.flatten()
    
    # Match sizes (if lengths differ)
    min_len = min(len(test_labels), len(anomaly_scores))
    test_labels = test_labels[:min_len]
    anomaly_scores = anomaly_scores[:min_len]
    
    # Debug information
    print(f"Test labels shape: {test_labels.shape}")
    print(f"Anomaly scores shape: {anomaly_scores.shape}")
    print(f"Test labels unique values: {np.unique(test_labels)}")
    print(f"Anomaly scores range: [{anomaly_scores.min():.4f}, {anomaly_scores.max():.4f}]")
    
    # Set threshold (90th percentile)
    threshold = np.percentile(anomaly_scores, 90)
    predictions = (anomaly_scores > threshold).astype(int)
    
    # Calculate performance metrics
    metrics = {}
    
    try:
        # Calculate AUC (check binary classification)
        if len(np.unique(test_labels)) == 2:
            metrics['auc'] = roc_auc_score(test_labels, anomaly_scores)
        else:
            print(f"Warning: Not binary classification. Unique labels: {np.unique(test_labels)}")
            metrics['auc'] = None
        
        # Other metrics
        metrics['f1'] = f1_score(test_labels, predictions, average='binary')
        metrics['precision'] = precision_score(test_labels, predictions, average='binary', zero_division=0)
        metrics['recall'] = recall_score(test_labels, predictions, average='binary', zero_division=0)
        metrics['accuracy'] = accuracy_score(test_labels, predictions)
        
        print(f"AUC: {metrics['auc']:.4f}" if metrics['auc'] else "AUC: N/A")
        print(f"F1 Score: {metrics['f1']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall: {metrics['recall']:.4f}")
        print(f"Accuracy: {metrics['accuracy']:.4f}")
        
    except Exception as e:
        print(f"Error in metrics calculation: {e}")
        metrics = {
            'auc': None,
            'f1': 0.0,
            'precision': 0.0,
            'recall': 0.0,
            'accuracy': 0.0
        }
    
    # Additional information
    metrics.update({
        'threshold': threshold,
        'anomaly_scores': anomaly_scores,
        'predictions': predictions,
        'test_labels': test_labels
    })
    
    return model, history, metrics

In [9]:
class ProbabilisticDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super().__init__()
        self.output_dim = output_dim
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim * 2)
        )
        
        # 가중치 초기화
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
    
    def forward(self, z):
        # 디코딩
        out = self.decoder(z)
        
        # 출력을 mean과 log variance로 분리
        mean, logvar = torch.chunk(out, 2, dim=-1)
        
        # log variance 클램핑
        logvar = torch.clamp(logvar, min=-20, max=20)
        
        return mean, logvar

In [10]:
# ODE-RNN/GRU-D + Time2Vec + Conditional Flow (최종)

class Time2Vec(nn.Module):
    def __init__(self, out_dim, num_frequencies=4):
        super().__init__()
        self.out_dim = out_dim
        self.num_frequencies = num_frequencies
        
        # Linear component
        self.linear = nn.Linear(1, 1)
        
        # Periodic components with multiple frequencies
        self.frequencies = nn.Parameter(torch.randn(num_frequencies) * 0.1)
        
        # 출력 차원을 정확히 맞추기 위해 계산
        periodic_dim = out_dim - 1  # 전체 차원에서 linear 차원(1)을 뺌
        self.periodic = nn.Linear(1, periodic_dim)
        
        print(f"Time2Vec initialized with:")
        print(f"- Total output dimension: {out_dim}")
        print(f"- Linear component dimension: 1")
        print(f"- Periodic component dimension: {periodic_dim}")
        
    def forward(self, t):
        # Linear component
        v1 = self.linear(t)  # (B, T, 1)
        
        # Periodic component
        periodic = torch.sin(self.frequencies[0] * self.periodic(t))  # (B, T, out_dim-1)
        
        # Concatenate and verify dimension
        output = torch.cat([v1, periodic], dim=-1)  # (B, T, out_dim)
        assert output.size(-1) == self.out_dim, \
            f"Expected output dimension {self.out_dim}, got {output.size(-1)}"
        
        return output

class ODERNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, time_emb_dim=8):
        super().__init__()
        self.time2vec = Time2Vec(time_emb_dim)
        
        # 입력 차원 계산 및 확인
        self.input_dim = input_dim
        self.time_emb_dim = time_emb_dim
        self.total_input_dim = input_dim + time_emb_dim
        
        print(f"Initializing ODERNNEncoder with:")
        print(f"- Input dimension: {input_dim}")
        print(f"- Time embedding dimension: {time_emb_dim}")
        print(f"- Total input dimension: {self.total_input_dim}")
        
        self.rnn = nn.GRU(self.total_input_dim, hidden_dim, batch_first=True)
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)
        
    def forward(self, x, times, mask=None):
        # 입력 차원 확인
        print(f"Input shape: {x.shape}")
        
        # 시간 임베딩
        t_emb = self.time2vec(times.unsqueeze(-1))  # (B, T, time_emb_dim)
        print(f"Time embedding shape: {t_emb.shape}")
        
        # 입력과 시간 임베딩 결합
        x_cat = torch.cat([x, t_emb], dim=-1)  # (B, T, input_dim + time_emb_dim)
        print(f"Concatenated shape: {x_cat.shape}")
        
        # 차원 검증
        assert x_cat.size(-1) == self.total_input_dim, \
            f"Expected input dimension {self.total_input_dim}, got {x_cat.size(-1)}"
        
        # RNN 처리
        h, _ = self.rnn(x_cat)  # (B, T, hidden_dim)
        h_last = h[:, -1]  # (B, hidden_dim)
        
        # 잠재 공간으로 매핑
        mu = self.mu(h_last)  # (B, latent_dim)
        logvar = self.logvar(h_last)  # (B, latent_dim)
        
        # 잠재 변수 샘플링
        z0 = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        
        return z0, mu, logvar

class IrregularTimeSeriesAnomalyDetector(nn.Module):
    def __init__(self, input_dim, latent_dim=32, hidden_dim=64, flow_layers=4, time_emb_dim=8, beta=0.1):
        super().__init__()
        
        # 차원이 짝수인지 확인
        if latent_dim % 2 != 0:
            latent_dim += 1
            print(f"Adjusted latent_dim to {latent_dim} to ensure it's even.")
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.time_emb_dim = time_emb_dim
        self.beta = beta
        self.temperature = 0.1  
        
        # 개선: 동적 anomaly_weight 설정
        self.anomaly_weight = 2.5  # 기본값을 낮춤 (5.0 → 2.5)
        self.adaptive_weight = True  # 적응적 가중치 사용 여부
        
        # 개선: 섹션별 가중치 저장
        self.section_weights = {
            'no_anomaly': 2.0,      # 이상치 없는 섹션
            'rare_anomaly': 8.0,    # 드문 이상치 (5% 미만)
            'moderate_anomaly': 5.0, # 보통 수준 (5-20%)
            'high_anomaly': 3.0     # 높은 이상치 (20% 이상)
        }
        
        # 인코더 (ODE-RNN + Time2Vec)
        self.encoder = ODERNNEncoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            time_emb_dim=time_emb_dim
        )
        
        # ODE 함수 (Time2Vec 포함)
        self.odefunc = ODEFunc(
            latent_dim=latent_dim,
            hidden_dim=hidden_dim,
            time_emb_dim=time_emb_dim
        )
        
        # 디코더
        self.decoder = ProbabilisticDecoder(
            latent_dim=latent_dim,
            output_dim=input_dim,
            hidden_dim=hidden_dim
        )
        
        # Flow 모델
        self.flow = SimpleFlow(dim=latent_dim, n_layers=flow_layers)
        
        # 사전 확률 분포 파라미터
        self.register_buffer('prior_mean', torch.zeros(latent_dim))
        self.register_buffer('prior_logvar', torch.zeros(latent_dim))
        
        # 개선: 메모리 모듈 추가 (정상 패턴 저장)
        self.memory_size = 100
        self.register_buffer('normal_memory', torch.randn(self.memory_size, latent_dim) * 0.1)
        self.memory_ptr = 0
        
    def set_anomaly_weight(self, weight):
        """anomaly_weight 설정"""
        self.anomaly_weight = weight
        print(f"anomaly_weight set to: {weight}")
    
    def set_section_weight(self, section_type):
        """섹션 유형에 따른 가중치 설정"""
        if section_type in self.section_weights:
            self.anomaly_weight = self.section_weights[section_type]
            print(f"Section type '{section_type}' detected, weight set to: {self.anomaly_weight}")
        else:
            print(f"Unknown section type: {section_type}, using default weight: {self.anomaly_weight}")
    
    def analyze_section_characteristics(self, test_labels):
        """섹션 특성 분석 및 적절한 가중치 설정"""
        if test_labels is None:
            return "unknown"
            
        total_samples = len(test_labels)
        anomaly_samples = np.sum(test_labels)
        anomaly_ratio = anomaly_samples / total_samples
        
        print(f"Section analysis: {total_samples} samples, {anomaly_samples} anomalies ({anomaly_ratio:.4f})")
        
        if anomaly_ratio == 0:
            section_type = "no_anomaly"
        elif anomaly_ratio < 0.05:
            section_type = "rare_anomaly"
        elif anomaly_ratio < 0.2:
            section_type = "moderate_anomaly"
        else:
            section_type = "high_anomaly"
        
        self.set_section_weight(section_type)
        return section_type
        
    def encode(self, x, times, mask=None):
        return self.encoder(x, times, mask)
        
    def decode(self, z0, times):
        if times.dim() > 1:
            times = times[0]
        
        times, indices = torch.sort(times)
        
        zt = odeint(
            self.odefunc,
            z0,
            times,
            method='dopri5',
            rtol=1e-3,
            atol=1e-4
        )
        
        zt = zt.permute(1, 0, 2)
        mu, logvar = self.decoder(zt)
        
        return mu, logvar
        
    def temporal_contrastive_loss(self, z_normal, z_augmented, times, temperature=0.1):
        """
        시간적 연속성을 고려한 대조 학습
        """
        # L2 정규화
        z_normal = F.normalize(z_normal, dim=-1)
        z_augmented = F.normalize(z_augmented, dim=-1)
        
        # 시간 차이에 따른 가중치 계산
        time_diff = torch.abs(times[:, 1:] - times[:, :-1])
        time_weights = torch.exp(-time_diff)  # 시간이 가까울수록 가중치가 큼
        
        # 유사도 계산
        similarity = torch.mm(z_normal, z_augmented.t()) / temperature
        
        # 시간적 연속성을 고려한 loss
        temporal_loss = 0
        for i in range(z_normal.size(0)-1):
            if i < time_weights.size(1):
                weight = time_weights[0, i]
                temporal_loss += weight * F.cross_entropy(
                    similarity[i:i+1], 
                    torch.tensor([i], device=z_normal.device)
                )
        
        return temporal_loss / (z_normal.size(0) - 1)

    def compute_loss(self, x, times, recon_mu, recon_logvar, mu, logvar, z_flow, log_det, mask=None):
        device = x.device
        batch_size = x.shape[0]
        
        # 기존 loss들
        if mask is not None:
            mask_expanded = mask.unsqueeze(-1).expand_as(x)
            recon_error = (recon_mu - x) ** 2
            recon_loss = (recon_error * mask_expanded).sum() / (mask_expanded.sum() + 1e-8)
        else:
            recon_loss = F.mse_loss(recon_mu, x, reduction='mean')
        
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size
        
        if z_flow is not None:
            flow_loss = 0.5 * torch.sum(z_flow**2) / batch_size
        else:
            flow_loss = torch.tensor(0.0, device=device)
        
        # 데이터 증강 및 시간적 대조 학습
        if self.training and batch_size > 1:
            # 노이즈 증강
            noise_factor = 0.01
            x_augmented = x + torch.randn_like(x) * noise_factor
            
            # 증강된 데이터의 잠재 표현
            z0_aug, _, _ = self.encode(x_augmented, times, mask)
            
            # 시간적 대조 학습 loss
            contrast_loss = self.temporal_contrastive_loss(mu, z0_aug, times)
        else:
            contrast_loss = torch.tensor(0.0, device=device)
        
        # 개선: 더 강화된 이상치 인식 손실
        anomaly_loss = self.compute_enhanced_anomaly_loss(x, recon_mu, recon_logvar, mu)

        total_loss = (
            recon_loss + 
            self.beta * kl_loss + 
            0.01 * flow_loss + 
            0.2 * contrast_loss +
            0.15 * anomaly_loss  # 가중치 증가 (0.1 → 0.15)
        )
        
        return {
            'total_loss': total_loss,
            'recon_loss': recon_loss,
            'kl_loss': kl_loss,
            'flow_loss': flow_loss,
            'contrast_loss': contrast_loss,
            'anomaly_loss': anomaly_loss
        }

    def compute_enhanced_anomaly_loss(self, x, recon_mu, recon_logvar, mu):
        """개선된 이상치 감지 손실 함수"""
        # 재구성 오차 계산
        recon_error = torch.abs(x - recon_mu)
        
        # 동적 임계값 계산 (더 민감하게)
        mean_error = torch.mean(recon_error, dim=(0, 1))
        std_error = torch.std(recon_error, dim=(0, 1))
        threshold = mean_error + 1.5 * std_error  # 2 → 1.5로 낮춤 (더 민감하게)
        
        # 이상치 가중치 계산 (더 세밀하게)
        anomaly_weights = torch.where(
            recon_error > threshold.unsqueeze(0).unsqueeze(0),
            torch.ones_like(recon_error) * self.anomaly_weight,
            torch.ones_like(recon_error) * 0.5  # 정상 데이터에 대한 가중치도 조정
        )
        
        # 가중치가 적용된 재구성 손실
        weighted_loss = torch.mean(recon_error * anomaly_weights)
        
        # 추가: 잠재 공간에서의 이상치 탐지
        latent_anomaly = torch.mean(torch.abs(mu), dim=-1)
        latent_threshold = torch.mean(latent_anomaly) + torch.std(latent_anomaly)
        latent_weights = torch.where(
            latent_anomaly > latent_threshold,
            torch.ones_like(latent_anomaly) * self.anomaly_weight,
            torch.ones_like(latent_anomaly)
        )
        latent_loss = torch.mean(latent_anomaly * latent_weights)
        
        return weighted_loss + 0.1 * latent_loss
    
    def forward(self, x, times, mask=None):
        if times.dim() == 1:
            times = times.unsqueeze(0).expand(x.size(0), -1)
        
        unique_times = torch.linspace(0, 1, times.size(1)).to(times.device)
        
        z0, mu, logvar = self.encode(x, times, mask)
        z_flow, log_det = self.flow(z0)
        recon_mu, recon_logvar = self.decode(z0, unique_times)
        
        losses = self.compute_loss(x, times, recon_mu, recon_logvar, mu, logvar, z_flow, log_det, mask)
        
        return {
            'z0': z0,
            'mu': mu,
            'logvar': logvar,
            'z_flow': z_flow,
            'log_det': log_det,
            'recon_mu': recon_mu,
            'recon_logvar': recon_logvar,
            'losses': losses
        }
        
    def anomaly_score(self, x, times, mask=None):
        self.eval()
        
        with torch.no_grad():
            outputs = self.forward(x, times, mask)
            
            # 1. 개선된 재구성 오차 (시계열 특성 고려)
            recon_error = torch.abs(x - outputs['recon_mu'])
            std = torch.exp(0.5 * outputs['recon_logvar']) + 1e-6
            
            # Point-wise anomaly score with temporal smoothing
            pointwise_scores = recon_error / std
            
            # Temporal smoothing (이동평균)
            kernel_size = min(5, x.shape[1])
            if kernel_size > 1:
                padding = kernel_size // 2
                pointwise_scores = F.avg_pool1d(
                    pointwise_scores.transpose(1, 2), 
                    kernel_size, stride=1, padding=padding
                ).transpose(1, 2)
            
            if mask is not None:
                mask_expanded = mask.unsqueeze(-1)
                pointwise_scores = pointwise_scores * mask_expanded
            
            # Feature-wise aggregation with temporal attention
            time_diff = torch.abs(times[:, 1:] - times[:, :-1])
            time_weights = torch.exp(-time_diff)  # 시간이 가까울수록 가중치가 큼
            time_weights = F.pad(time_weights, (0, 1), value=1.0)
            
            # 5. 특성별 점수 계산
            feature_scores = pointwise_scores * time_weights.unsqueeze(-1)
            
            # 6. 시퀀스 레벨 통계 계산
            max_scores = torch.max(feature_scores, dim=1)[0]  # (B, F)
            mean_scores = torch.mean(feature_scores, dim=1)   # (B, F)
            std_scores = torch.std(feature_scores, dim=1)     # (B, F)
            
            # 7. KL 발산 점수 계산
            kl_scores = 0.5 * torch.sum(
                outputs['mu'].pow(2) + torch.exp(outputs['logvar']) - outputs['logvar'] - 1,
                dim=-1
            ).unsqueeze(-1)

            # 3. Flow-based density score
            flow_scores = 0.5 * torch.sum(outputs['z_flow']**2, dim=-1).unsqueeze(-1)  # (B, 1)
            
            # 개선: 더 민감한 동적 임계값
            dynamic_threshold = mean_scores + 1.5 * std_scores  # 2 → 1.5로 낮춤

            # 개선: 가중치 조정
            final_scores = (
                0.40 * max_scores +     # 최대 재구성 오차 (0.35 → 0.40)
                0.25 * mean_scores +    # 평균 재구성 오차
                0.15 * std_scores +     # 재구성 오차 변동성
                0.10 * kl_scores.expand(-1, self.input_dim) +  # KL 발산 (0.15 → 0.10)
                0.10 * flow_scores.expand(-1, self.input_dim)  # Flow 밀도
            )
            
            # 개선: 더 세밀한 이상치 마스킹
            anomaly_mask = final_scores > dynamic_threshold
            final_scores = torch.where(
                anomaly_mask,
                final_scores * self.anomaly_weight,
                final_scores * 0.8  # 정상 데이터에 대한 가중치도 조정
            )
            
            # 12. 시퀀스 레벨 점수로 변환
            sequence_scores = torch.mean(final_scores, dim=-1)  # (B,)
            
            # 13. 점수 정규화 (개선된 방식)
            min_score = sequence_scores.min()
            max_score = sequence_scores.max()
            if max_score > min_score:
                sequence_scores = (sequence_scores - min_score) / (max_score - min_score)
            else:
                sequence_scores = torch.zeros_like(sequence_scores)
            
            return sequence_scores.cpu().numpy()

class ODERNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, time_emb_dim=8):
        super().__init__()
        self.time2vec = Time2Vec(time_emb_dim)
        self.rnn = nn.GRU(input_dim + time_emb_dim, hidden_dim, batch_first=True)
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)
    def forward(self, x, times, mask=None):
        t_emb = self.time2vec(times.unsqueeze(-1))  # (B, T, time_emb_dim)
        x_cat = torch.cat([x, t_emb], dim=-1)
        h, _ = self.rnn(x_cat)
        h_last = h[:, -1]
        mu = self.mu(h_last)
        logvar = self.logvar(h_last)
        z0 = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return z0, mu, logvar

class ODEFunc(nn.Module):
    def __init__(self, latent_dim, hidden_dim, time_emb_dim=8):
        super().__init__()
        self.time2vec = Time2Vec(time_emb_dim)
        
        # 시간 임베딩과 상태를 결합한 네트워크
        self.net = nn.Sequential(
            nn.Linear(latent_dim + time_emb_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, latent_dim)
        )
        
    def forward(self, t, z):
        # 시간 임베딩
        t_tensor = t.reshape(1, 1).repeat(z.size(0), 1)
        t_emb = self.time2vec(t_tensor)
        
        # 상태와 시간 임베딩 결합
        z_t = torch.cat([z, t_emb], dim=1)
        
        # 속도장 계산
        dz_dt = self.net(z_t)
        return dz_dt

class ProbabilisticDecoder(nn.Module):
    def __init__(self, latent_dim, output_dim, hidden_dim):
        super().__init__()
        # 더 깊은 네트워크와 정규화 추가
        self.decoder_net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),  # Layer Normalization 추가
            nn.ReLU(),
            nn.Dropout(0.1),  # Dropout 추가
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
        )
        
        # 출력 레이어 분리 및 초기화 개선
        self.mu_net = nn.Linear(hidden_dim, output_dim)
        self.logvar_net = nn.Linear(hidden_dim, output_dim)
        
        # 가중치 초기화
        self._init_weights()

    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, z):
        h = self.decoder_net(z)
        mu = self.mu_net(h)
        logvar = self.logvar_net(h)
        # 분산의 범위를 더 넓게 설정
        logvar = torch.clamp(logvar, -20, 2)
        return mu, logvar
    

class SimpleFlow(nn.Module):
    def __init__(self, dim, n_layers=2):
        super().__init__()
        if dim % 2 != 0:
            raise ValueError(f"Dimension must be even, got {dim}")
            
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(AffineCoupling(dim))
            
    def forward(self, z):
        log_det = torch.zeros(z.size(0), device=z.device)
        for layer in self.layers:
            z, ldj = layer(z)
            log_det += ldj
        return z, log_det

class AffineCoupling(nn.Module):
    def __init__(self, dim, hidden_dim=64):
        super().__init__()
        self.dim = dim
        assert dim % 2 == 0, "dim must be even"
        
        self.net = nn.Sequential(
            nn.Linear(dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, dim)
        )
    
    def forward(self, x, reverse=False):
        x1, x2 = torch.chunk(x, 2, dim=1)
        h = self.net(x1)
        shift, scale = torch.chunk(h, 2, dim=1)
        scale = torch.tanh(scale)
        
        if not reverse:
            y2 = x2 * torch.exp(scale) + shift
        else:
            y2 = (x2 - shift) * torch.exp(-scale)
            
        y = torch.cat([x1, y2], dim=1)
        log_det = scale.sum(dim=1)
        
        if reverse:
            log_det = -log_det
            
        return y, log_det

In [11]:
class ContinuousMemoryBank(nn.Module):
    def __init__(self, latent_dim, memory_size=100):
        super().__init__()
        self.latent_dim = latent_dim
        self.memory_size = memory_size
        
        self.register_buffer('memory_trajectories', torch.randn(memory_size, latent_dim) * 0.1)
        self.register_buffer('memory_ptr', torch.zeros(1, dtype=torch.long))
        
    def update_memory(self, z_trajectory):
        if not self.training:
            return
            
        # detach하여 그래디언트 그래프에서 분리
        z_mean = torch.mean(z_trajectory.detach(), dim=0)
        
        for i in range(z_mean.size(0)):
            ptr = self.memory_ptr.item()
            self.memory_trajectories[ptr] = z_mean[i].detach()
            self.memory_ptr[0] = (ptr + 1) % self.memory_size
    
    def compute_memory_distance(self, z_trajectory):
        z_mean = torch.mean(z_trajectory, dim=0)
        
        distances = []
        for i in range(z_mean.size(0)):
            z_sample = z_mean[i].unsqueeze(0)
            # memory_trajectories는 이미 detached된 버퍼이므로 안전
            dists = torch.cdist(z_sample, self.memory_trajectories.detach())
            min_dist = torch.min(dists)
            distances.append(min_dist)
        
        return torch.stack(distances)

class TrajectoryAnomalyScorer(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.trajectory_net = nn.Sequential(
            nn.Linear(latent_dim * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def compute_trajectory_features(self, z_trajectory):
        velocity = torch.diff(z_trajectory, dim=0)
        velocity_mean = torch.mean(velocity, dim=0)
        
        z_mean = torch.mean(z_trajectory, dim=0)
        combined = torch.cat([z_mean, velocity_mean], dim=-1)
        
        return combined
    
    def forward(self, z_trajectory):
        trajectory_features = self.compute_trajectory_features(z_trajectory)
        trajectory_scores = self.trajectory_net(trajectory_features).squeeze(-1)
        return trajectory_scores



class Time2Vec(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.w = nn.Parameter(torch.randn(embedding_dim))
        self.b = nn.Parameter(torch.randn(embedding_dim))
        
    def forward(self, t):
        if t.dim() == 2:
            t = t.unsqueeze(-1)
        elif t.dim() == 1:
            t = t.unsqueeze(-1).unsqueeze(-1)
        
        t_emb = torch.sin(t * self.w.view(1, 1, -1) + self.b.view(1, 1, -1))
        
        if t_emb.dim() == 3:
            t_emb = t_emb.squeeze(-2)
        
        return t_emb

class ODERNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, time_emb_dim=8):
        super().__init__()
        self.time2vec = Time2Vec(time_emb_dim)
        self.rnn = nn.GRU(input_dim + time_emb_dim, hidden_dim, batch_first=True)
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)
        
    def forward(self, x, times, mask=None):
        t_emb = self.time2vec(times)
        x_cat = torch.cat([x, t_emb], dim=-1)
        h, _ = self.rnn(x_cat)
        h_last = h[:, -1]
        mu = self.mu(h_last)
        logvar = self.logvar(h_last)
        z0 = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return z0, mu, logvar

class ODEFunc(nn.Module):
    def __init__(self, latent_dim, hidden_dim, time_emb_dim=8):
        super().__init__()
        self.time2vec = Time2Vec(time_emb_dim)
        
        self.net = nn.Sequential(
            nn.Linear(latent_dim + time_emb_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, latent_dim)
        )
        
    def forward(self, t, z):
        batch_size = z.size(0)
        
        if isinstance(t, (int, float)):
            t_val = t
        else:
            t_val = t.item() if t.numel() == 1 else t[0].item()
        
        t_tensor = torch.full((batch_size, 1), t_val, dtype=z.dtype, device=z.device)
        t_emb = self.time2vec(t_tensor)
        
        if t_emb.dim() == 3:
            t_emb = t_emb.squeeze(1)
        
        z_t = torch.cat([z, t_emb], dim=1)
        dz_dt = self.net(z_t)
        return dz_dt

class ProbabilisticDecoder(nn.Module):
    def __init__(self, latent_dim, output_dim, hidden_dim):
        super().__init__()
        self.decoder_net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
        )
        
        self.mu_net = nn.Linear(hidden_dim, output_dim)
        self.logvar_net = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, z):
        h = self.decoder_net(z)
        mu = self.mu_net(h)
        logvar = self.logvar_net(h)
        logvar = torch.clamp(logvar, -20, 2)
        return mu, logvar

class EnhancedNeuralODEAnomalyDetector(nn.Module):
    def __init__(self, input_dim, latent_dim=32, hidden_dim=64, flow_layers=6, time_emb_dim=8, beta=0.1, memory_size=100):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.time_emb_dim = time_emb_dim
        self.beta = beta
        
        self.encoder = ODERNNEncoder(input_dim, hidden_dim, latent_dim, time_emb_dim)
        self.ode_func = ODEFunc(latent_dim, hidden_dim, time_emb_dim)
        self.decoder = ProbabilisticDecoder(latent_dim, input_dim, hidden_dim)
        
        self.memory_bank = ContinuousMemoryBank(latent_dim, memory_size)
        self.trajectory_scorer = TrajectoryAnomalyScorer(latent_dim)
        
        self.anomaly_weight = 2.5
        self.adaptive_weight = True
        
        self.section_weights = {
            'no_anomaly': 1.5,
            'rare_anomaly': 120,
            'moderate_anomaly': 8.0,
            'high_anomaly': 4.0
        }
    
    def set_anomaly_weight(self, weight):
        self.anomaly_weight = weight
        print(f"anomaly_weight set to: {weight}")
    
    def set_section_weight(self, section_type):
        if section_type in self.section_weights:
            self.anomaly_weight = self.section_weights[section_type]
            print(f"Section type '{section_type}' detected, weight set to: {self.anomaly_weight}")
        else:
            print(f"Unknown section type: {section_type}, using default weight: {self.anomaly_weight}")
        
    def forward(self, x, times, mask=None):
        if times.dim() == 1:
            times = times.unsqueeze(0).expand(x.size(0), -1)
        
        z0, mu, logvar = self.encoder(x, times, mask)
        
        if times.dim() > 1:
            unique_times = torch.linspace(0, 1, times.size(1)).to(times.device)
        else:
            unique_times = times
            
        try:
            z_trajectory = odeint(
                self.ode_func, 
                z0, 
                unique_times, 
                method='dopri5',
                rtol=1e-3,
                atol=1e-4
            )
        except Exception as e:
            print(f"ODE integration failed, using simpler method: {e}")
            z_trajectory = odeint(self.ode_func, z0, unique_times, method='euler')
        
        if self.training:
            self.memory_bank.update_memory(z_trajectory)
        
        mu_recon, logvar_recon = self.decoder(z_trajectory)
        
        losses = self.compute_enhanced_loss(x, {
            'mu_recon': mu_recon,
            'mu': mu,
            'logvar': logvar,
            'z_trajectory': z_trajectory
        }, mask)
        
        return {
            'z0': z0,
            'mu': mu,
            'logvar': logvar,
            'z_trajectory': z_trajectory,
            'mu_recon': mu_recon,
            'logvar_recon': logvar_recon,
            'losses': losses
        }
    
    def compute_enhanced_loss(self, x, outputs, mask=None):
        mu_recon = outputs['mu_recon']
        mu = outputs['mu']
        logvar = outputs['logvar']
        z_trajectory = outputs['z_trajectory']
        
        x_expanded = x.transpose(0, 1)
        recon_loss = F.mse_loss(mu_recon, x_expanded, reduction='mean')
        
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
        
        memory_distances = self.memory_bank.compute_memory_distance(z_trajectory)
        memory_loss = torch.mean(memory_distances)
        
        trajectory_scores = self.trajectory_scorer(z_trajectory)
        trajectory_loss = torch.mean(trajectory_scores ** 2)
        
        total_loss = (recon_loss + 
                     self.beta * kl_loss + 
                     0.05 * memory_loss + 
                     0.02 * trajectory_loss)
        
        return {
            'total_loss': total_loss,
            'recon_loss': recon_loss,
            'kl_loss': kl_loss,
            'memory_loss': memory_loss,
            'trajectory_loss': trajectory_loss
        }
    
    def anomaly_score(self, x, times, mask=None):
        self.eval()
        
        # 디바이스 일관성 보장
        device = next(self.parameters()).device
        
        # 모든 서브모듈을 같은 디바이스로 강제 이동
        for module in self.modules():
            module.to(device)
        
        # 입력 텐서들을 모델과 같은 디바이스로 이동
        if isinstance(x, torch.Tensor):
            x = x.to(device)
        else:
            x = torch.tensor(x, dtype=torch.float32, device=device)
            
        if isinstance(times, torch.Tensor):
            times = times.to(device)
        else:
            times = torch.tensor(times, dtype=torch.float32, device=device)
            
        if mask is not None:
            if isinstance(mask, torch.Tensor):
                mask = mask.to(device)
            else:
                mask = torch.tensor(mask, dtype=torch.float32, device=device)
        
        with torch.no_grad():
            outputs = self.forward(x, times, mask)
            z_trajectory = outputs['z_trajectory']
            
            mu_recon = outputs['mu_recon']
            x_expanded = x.transpose(0, 1)
            recon_errors = torch.mean((mu_recon - x_expanded) ** 2, dim=(0, 2))
            
            memory_distances = self.memory_bank.compute_memory_distance(z_trajectory)
            
            trajectory_scores = self.trajectory_scorer(z_trajectory)
            
            final_scores = (0.6 * recon_errors + 
                          0.2 * memory_distances + 
                          0.2 * trajectory_scores)
            
            min_score = final_scores.min()
            max_score = final_scores.max()
            if max_score > min_score:
                final_scores = (final_scores - min_score) / (max_score - min_score)
            
            return final_scores

In [34]:
### Model version 3 (gradient and ema base robust method)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchdiffeq import odeint
import numpy as np

class GradientAwareInitialization:
    @staticmethod
    def init_weights_by_gradient_range(model, data_sample, gradient_scale=0.1):
        """
        데이터의 미분값 범위를 기반으로 가중치 초기화
        """
        with torch.no_grad():
            # 샘플 데이터에서 gradient 범위 추정
            x_sample = data_sample[:5]  # 작은 샘플 사용
            gradients = torch.diff(x_sample, dim=1)
            grad_std = torch.std(gradients)
            grad_mean = torch.mean(torch.abs(gradients))
            
            # 가중치 스케일 계산
            weight_scale = gradient_scale / (grad_std + 1e-8)
            
            for name, param in model.named_parameters():
                if 'weight' in name:
                    if 'ode' in name.lower():
                        # ODE 함수는 더 작은 초기값
                        nn.init.normal_(param, 0, weight_scale * 0.1)
                    elif 'gradient' in name.lower():
                        # Gradient 관련 레이어는 적응적 초기화
                        nn.init.normal_(param, 0, weight_scale)
                    else:
                        # 일반 레이어는 Xavier 초기화
                        nn.init.xavier_normal_(param, gain=weight_scale)
                elif 'bias' in name:
                    nn.init.zeros_(param)
            
            print(f"Initialized weights with gradient_scale: {weight_scale:.6f}")

class RobustTemporalGradientAnalyzer(nn.Module):
    def __init__(self, latent_dim, max_gradient_norm=1.0, smoothing_factor=0.1):
        super().__init__()
        self.max_gradient_norm = max_gradient_norm
        self.smoothing_factor = smoothing_factor
        
        # Gradient processing network
        self.gradient_processor = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2),
            nn.LayerNorm(latent_dim * 2),
            nn.Tanh(),
            nn.Linear(latent_dim * 2, latent_dim)
        )
        
        # Adaptive gradient scaling
        self.gradient_scale = nn.Parameter(torch.tensor(1.0))
        
        # Moving average for gradient statistics
        self.register_buffer('grad_ema_mean', torch.zeros(1))
        self.register_buffer('grad_ema_var', torch.ones(1))
        
    def compute_robust_gradients(self, z_trajectory, times):
        """
        Robust gradient computation with constraints
        """
        # 시간에 대한 미분 계산
        dz_dt = torch.diff(z_trajectory, dim=0)  # (T-1, B, D)
        
        # 시간 차분 계산
        if times.dim() > 0 and len(times) > 1:
            time_diffs = torch.diff(times)  # (T-1,)
            time_diffs = torch.clamp(time_diffs, min=1e-6)
            
            # 차원 맞추기
            if dz_dt.dim() == 3:  # (T-1, B, D)
                time_diffs = time_diffs.unsqueeze(-1).unsqueeze(-1)  # (T-1, 1, 1)
            elif dz_dt.dim() == 2:  # (T-1, D)
                time_diffs = time_diffs.unsqueeze(-1)  # (T-1, 1)
            
            dz_dt = dz_dt / time_diffs
        
        # 배치 차원이 있는 경우 평균 사용
        if dz_dt.dim() == 3:  # (T-1, B, D)
            dz_dt = dz_dt.mean(dim=1)  # (T-1, D)
        
        # Gradient clipping for stability
        dz_dt = torch.clamp(dz_dt, -self.max_gradient_norm, self.max_gradient_norm)
        
        # EMA 업데이트 시 detach() 사용하여 gradient graph 분리
        if self.training and len(dz_dt) > 0:
            with torch.no_grad():  # gradient 계산 방지
                current_mean = torch.mean(torch.abs(dz_dt.detach()))
                current_var = torch.var(torch.abs(dz_dt.detach()))
                
                self.grad_ema_mean = (1 - self.smoothing_factor) * self.grad_ema_mean + \
                                    self.smoothing_factor * current_mean
                self.grad_ema_var = (1 - self.smoothing_factor) * self.grad_ema_var + \
                                   self.smoothing_factor * current_var
        
        # Normalize gradients (detached EMA 값 사용)
        if len(dz_dt) > 0:
            normalized_gradients = (dz_dt - self.grad_ema_mean.detach()) / (torch.sqrt(self.grad_ema_var.detach()) + 1e-8)
            
            # Process through network
            processed_gradients = self.gradient_processor(normalized_gradients)
            
            return processed_gradients * self.gradient_scale
        else:
            return torch.zeros(0, dz_dt.size(-1), device=dz_dt.device)
    
    def compute_spike_indicators(self, gradients):
        """
        Compute different types of spike indicators
        """
        if len(gradients) == 0:
            device = next(self.parameters()).device
            return torch.zeros(1, device=device), torch.zeros(1, device=device)
        
        # Sudden spike indicator
        grad_norms = torch.norm(gradients, dim=-1)
        if len(grad_norms) > 1:
            grad_std = torch.std(grad_norms)
            sudden_spike = torch.mean(grad_norms * torch.exp(-grad_std))
        else:
            sudden_spike = torch.mean(grad_norms)
        
        # Gradual spike indicator
        if len(gradients) > 1:
            grad_direction = F.normalize(gradients, dim=-1)
            direction_consistency = torch.mean(torch.abs(torch.sum(grad_direction, dim=0)))
            gradual_spike = direction_consistency * torch.mean(grad_norms)
        else:
            gradual_spike = torch.mean(grad_norms)
        
        return sudden_spike, gradual_spike

class RobustGradientSpikeDetector(nn.Module):
    def __init__(self, input_dim, latent_dim=32, hidden_dim=64, time_emb_dim=8, beta=0.1):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.beta = beta
        
        # Encoder
        self.encoder = ODERNNEncoder(input_dim, hidden_dim, latent_dim, time_emb_dim)
        
        # ODE function
        self.ode_func = ODEFunc(latent_dim, hidden_dim, time_emb_dim)
        
        # Decoder
        self.decoder = ProbabilisticDecoder(latent_dim, input_dim, hidden_dim)
        
        # Robust gradient analyzer
        self.gradient_analyzer = RobustTemporalGradientAnalyzer(latent_dim)
        
        # Spike classifier
        self.spike_classifier = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2)
        )
        
    def forward(self, x, times, mask=None):
        if times.dim() == 1:
            times = times.unsqueeze(0).expand(x.size(0), -1)
        
        # Encode
        z0, mu, logvar = self.encoder(x, times, mask)
        
        # Prepare time vector for ODE
        seq_len = x.size(1)
        unique_times = torch.linspace(0, 1, seq_len).to(x.device)
        
        # ODE integration with error handling
        try:
            z_trajectory = odeint(
                self.ode_func, 
                z0, 
                unique_times, 
                method='dopri5',
                rtol=1e-4,
                atol=1e-6,
                options={'max_num_steps': 1000}
            )
        except Exception as e:
            # Fallback to simpler method
            z_trajectory = odeint(
                self.ode_func, 
                z0, 
                unique_times, 
                method='euler'
            )
        
        # Robust gradient analysis
        gradients = self.gradient_analyzer.compute_robust_gradients(z_trajectory, unique_times)
        sudden_spikes, gradual_spikes = self.gradient_analyzer.compute_spike_indicators(gradients)
        
        # Spike classification
        if len(z_trajectory) > 0:
            z_for_classification = z_trajectory.mean(dim=0)  # (B, D)
            spike_scores = self.spike_classifier(z_for_classification)
        else:
            spike_scores = torch.zeros(x.size(0), 2, device=x.device)
        
        # Reconstruction
        mu_recon, logvar_recon = self.decoder(z_trajectory)
        
        # Compute losses
        losses = self.compute_robust_loss(x, {
            'mu_recon': mu_recon,
            'logvar_recon': logvar_recon,
            'z_trajectory': z_trajectory,
            'gradients': gradients,
            'mu': mu,
            'logvar': logvar
        }, mask)
        
        return {
            'mu_recon': mu_recon,
            'logvar_recon': logvar_recon,
            'z_trajectory': z_trajectory,
            'gradients': gradients,
            'sudden_spikes': sudden_spikes,
            'gradual_spikes': gradual_spikes,
            'spike_scores': spike_scores,
            'mu': mu,
            'logvar': logvar,
            'losses': losses
        }
    
    def compute_robust_loss(self, x, outputs, mask=None):
        """
        Robust loss with gradient constraints
        """
        mu_recon = outputs['mu_recon']
        mu = outputs['mu']
        logvar = outputs['logvar']
        gradients = outputs['gradients']
        
        # Reconstruction loss
        x_expanded = x.transpose(0, 1)
        
        # 차원 맞추기
        min_len = min(mu_recon.size(0), x_expanded.size(0))
        mu_recon_trimmed = mu_recon[:min_len]
        x_expanded_trimmed = x_expanded[:min_len]
        
        recon_loss = F.mse_loss(mu_recon_trimmed, x_expanded_trimmed, reduction='mean')
        
        # KL divergence
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
        
        # Gradient smoothness loss
        if len(gradients) > 1:
            grad_smoothness = torch.mean(torch.diff(gradients, dim=0) ** 2)
        else:
            grad_smoothness = torch.tensor(0.0, device=x.device)
        
        # Total loss
        total_loss = recon_loss + self.beta * kl_loss + 0.01 * grad_smoothness
        
        return {
            'total_loss': total_loss,
            'recon_loss': recon_loss,
            'kl_loss': kl_loss,
            'grad_smoothness': grad_smoothness
        }
    
    def anomaly_score(self, x, times, mask=None):
        self.eval()
        with torch.no_grad():
            if times.dim() == 1:
                times = times.unsqueeze(0).expand(x.size(0), -1)
            
            z0, mu, logvar = self.encoder(x, times, mask)
            seq_len = x.size(1)
            unique_times = torch.linspace(0, 1, seq_len).to(x.device)
            
            try:
                z_trajectory = odeint(self.ode_func, z0, unique_times, method='dopri5',
                                    rtol=1e-4, atol=1e-6, options={'max_num_steps': 1000})
            except:
                z_trajectory = odeint(self.ode_func, z0, unique_times, method='euler')
            
            gradients = self.gradient_analyzer.compute_robust_gradients(z_trajectory, unique_times)
            sudden_spikes, gradual_spikes = self.gradient_analyzer.compute_spike_indicators(gradients)
            
            mu_recon, logvar_recon = self.decoder(z_trajectory)
            
            # Reconstruction error - 배치별로 개별 계산
            x_expanded = x.transpose(0, 1)  # (seq_len, batch, features)
            min_len = min(mu_recon.size(0), x_expanded.size(0))
            
            # 각 샘플별 reconstruction error 계산 (배치 차원 유지)
            recon_errors = torch.mean((mu_recon[:min_len] - x_expanded[:min_len]) ** 2, dim=(0, 2))  # (batch,)
            
            # Combined score - 각 샘플별로 계산
            if len(gradients) > 0:
                # Gradient anomaly를 각 샘플별로 계산
                grad_norms = torch.norm(gradients, dim=-1)  # (seq_len-1,)
                grad_anomaly = torch.mean(grad_norms)  # 스칼라
                
                # KL divergence 각 샘플별로 계산|
                kl_divs = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1)  # (batch,)
                
                # 각 샘플별 최종 점수 계산
                final_scores = (0.5 * recon_errors +           # reconstruction error (batch,)
                            0.2 * grad_anomaly +            # gradient magnitude (broadcast)
                            0.15 * sudden_spikes +          # sudden changes (broadcast)
                            0.1 * gradual_spikes +          # gradual changes (broadcast)
                            0.05 * kl_divs)                 # uncertainty (batch,)
            else:
                final_scores = recon_errors  # 각 샘플별 reconstruction error
            
            return final_scores.cpu().numpy()


### Open dataset

1. SMD

In [13]:
### SMD 
import kagglehub
import shutil
import os
from datasets.SMD.preprocess_SMD import SMD_Dataset  

In [21]:
# Download latest version
"""
path = kagglehub.dataset_download("mgusat/smd-onmiad")

print("Path to dataset files:", path)

# 현재 경로 (KaggleHub 캐시)
source_path = "/Users/jieun/.cache/kagglehub/datasets/mgusat/smd-onmiad/versions/1"
# 원하는 경로 (예: ./data/SMD)
destination_path = "./datasets/SMD"

os.makedirs(destination_path, exist_ok=True)

for item in os.listdir(source_path):
    s = os.path.join(source_path, item)
    d = os.path.join(destination_path, item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)

print("✅ SMD 데이터 복사 완료:", destination_path)
"""

'\npath = kagglehub.dataset_download("mgusat/smd-onmiad")\n\nprint("Path to dataset files:", path)\n\n# 현재 경로 (KaggleHub 캐시)\nsource_path = "/Users/jieun/.cache/kagglehub/datasets/mgusat/smd-onmiad/versions/1"\n# 원하는 경로 (예: ./data/SMD)\ndestination_path = "./datasets/SMD"\n\nos.makedirs(destination_path, exist_ok=True)\n\nfor item in os.listdir(source_path):\n    s = os.path.join(source_path, item)\n    d = os.path.join(destination_path, item)\n    if os.path.isdir(s):\n        shutil.copytree(s, d, dirs_exist_ok=True)\n    else:\n        shutil.copy2(s, d)\n\nprint("✅ SMD 데이터 복사 완료:", destination_path)\n'

In [14]:
def prepare_smd_data(x_trn, x_tst, lab_tst, batch_size=32):
    # 데이터 형태 확인
    print(f"Original data shapes:")
    print(f"x_trn: {x_trn.shape}, x_tst: {x_tst.shape}, lab_tst: {lab_tst.shape}")
    
    # 데이터 정규화 (Min-Max 스케일링)
    def min_max_scaler(data):
        data_reshaped = data.reshape(-1, data.shape[-1])
        min_vals = np.min(data_reshaped, axis=0)
        max_vals = np.max(data_reshaped, axis=0)
        scaled = (data_reshaped - min_vals) / (max_vals - min_vals + 1e-8)
        return scaled.reshape(data.shape), min_vals, max_vals
    
    # 학습 데이터 정규화
    x_trn_norm, min_vals, max_vals = min_max_scaler(x_trn)
    x_tst_norm = (x_tst.reshape(-1, x_tst.shape[-1]) - min_vals) / (max_vals - min_vals + 1e-8)
    x_tst_norm = x_tst_norm.reshape(x_tst.shape)
    
    # 데이터가 2D인 경우 3D로 변환 (시계열 윈도우 생성)
    if len(x_trn.shape) == 2:  # (samples, features)
        print("Converting 2D data to 3D windows...")
        window_size = min(50, x_trn.shape[0] // 10)  # 적절한 윈도우 크기 설정
        
        # 윈도우 생성 함수
        def create_windows(data, window_size, stride=1):
            windows = []
            for i in range(0, len(data) - window_size + 1, stride):
                windows.append(data[i:i + window_size])
            return np.array(windows)
        
        # 학습 데이터 윈도우 생성
        x_trn_windowed = create_windows(x_trn_norm, window_size)
        
        # 테스트 데이터 윈도우 생성
        x_tst_windowed = create_windows(x_tst_norm, window_size)
        
        # 라벨도 윈도우에 맞게 조정 (윈도우 내 최대값 사용)
        if len(lab_tst.shape) == 1:
            lab_tst_windowed = []
            for i in range(0, len(lab_tst) - window_size + 1):
                lab_tst_windowed.append(np.max(lab_tst[i:i + window_size]))
            lab_tst_windowed = np.array(lab_tst_windowed)
        else:
            lab_tst_windowed = create_windows(lab_tst, window_size)
            lab_tst_windowed = np.max(lab_tst_windowed, axis=1)
        
        # 업데이트된 데이터 사용
        x_trn_norm = x_trn_windowed
        x_tst_norm = x_tst_windowed
        lab_tst = lab_tst_windowed
        
        print(f"After windowing:")
        print(f"x_trn: {x_trn_norm.shape}, x_tst: {x_tst_norm.shape}, lab_tst: {lab_tst.shape}")
    
    # 텐서 변환
    x_trn_tensor = torch.FloatTensor(x_trn_norm)
    x_tst_tensor = torch.FloatTensor(x_tst_norm)
    lab_tst_tensor = torch.FloatTensor(lab_tst)
    
    def create_time_vector(seq_length):
        return torch.linspace(0, 1, seq_length)
    
    # 시간 벡터 생성 (3D 데이터 기준)
    if len(x_trn_tensor.shape) == 3:  # (batch, seq_len, features)
        train_times = create_time_vector(x_trn_tensor.shape[1])
        train_times = train_times.unsqueeze(0).expand(x_trn_tensor.shape[0], -1)
        
        test_times = create_time_vector(x_tst_tensor.shape[1])
        test_times = test_times.unsqueeze(0).expand(x_tst_tensor.shape[0], -1)
        
        # 마스크 생성 - 올바른 차원으로
        train_mask = torch.ones_like(x_trn_tensor[:, :, 0])  # (batch, seq_len)
        test_mask = torch.ones_like(x_tst_tensor[:, :, 0])   # (batch, seq_len)
        
    else:  # 2D 데이터인 경우
        raise ValueError(f"Data should be 3D after processing, but got shape {x_trn_tensor.shape}")
    
    # 데이터셋 생성
    train_dataset = TensorDataset(x_trn_tensor, train_times, train_mask)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_dataset = TensorDataset(x_tst_tensor, test_times, test_mask)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    scaler_params = {
        'min_vals': min_vals,
        'max_vals': max_vals
    }
    
    return {
        'train_loader': train_loader,
        "test_loader": test_loader,
        'x_tst': x_tst_tensor,
        'train_times': train_times,
        'test_times': test_times,
        'train_mask': train_mask,
        'test_mask': test_mask,
        'lab_tst': lab_tst_tensor,
        'scaler_params': scaler_params 
    }
    

def train_and_evaluate_smd(model, data_dict, num_epochs=50):
    """SMD 데이터를 위한 안전한 학습 및 평가 함수"""
    
    train_loader = data_dict['train_loader']
    x_tst = data_dict['x_tst']
    lab_tst = data_dict['lab_tst']
    
    device = next(model.parameters()).device
    
    # 옵티마이저 설정
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    print(f"=== 학습 시작 (총 {num_epochs} 에포크) ===")
    
    # 학습 루프
    model, history = train_irregular_ode(
        model,
        train_loader,
        optimizer,
        scheduler,
        num_epochs
    )
    
    print("=== 학습 완료 ===")
    
    # 평가
    print("=== 모델 평가 시작 ===")
    
    # 시퀀스 레벨 라벨 생성
    if lab_tst.ndim == 2:
        sequence_labels = (lab_tst.max(axis=1) > 0).astype(int)
    else:
        sequence_labels = lab_tst
    
    # 평가 수행
    metrics = simple_evaluate_model(model, x_tst, sequence_labels, "Enhanced_Neural_ODE")
    
    return model, history, metrics

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [40]:
def train_enhanced_model(model, train_loader, optimizer, device):
    model.train()
    total_loss = 0
    
    for batch_data in train_loader:
        if len(batch_data) == 2:
            batch_x, batch_times = batch_data
            batch_mask = None
        elif len(batch_data) == 3:
            batch_x, batch_times, batch_mask = batch_data
        else:
            raise ValueError(f"Unexpected number of items in batch: {len(batch_data)}")
        
        batch_x = batch_x.to(device)
        batch_times = batch_times.to(device)
        if batch_mask is not None:
            batch_mask = batch_mask.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(batch_x, batch_times, batch_mask)
        
        losses = model.compute_enhanced_loss(batch_x, outputs, batch_mask)
        loss = losses['total_loss']
        
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # 손실 값을 저장 (detach하여 그래프에서 분리)
        total_loss += loss.detach().item()
    
    return total_loss / len(train_loader)

def train_irregular_ode(model, train_loader, optimizer, scheduler, num_epochs=100):
    history = {'train_loss': []}
    
    for epoch in range(num_epochs):
        epoch_loss = train_enhanced_model(model, train_loader, optimizer, model.device if hasattr(model, 'device') else next(model.parameters()).device)
        history['train_loss'].append(epoch_loss)
        
        if scheduler:
            scheduler.step(epoch_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.6f}")
    
    return model, history

In [35]:
def simple_evaluate_model(model, test_data, test_labels, section_id="SWaT_Section"):
    print(f"\n=== {section_id} 성능 평가 ===")
    
    device = next(model.parameters()).device
    model.eval()
    
    with torch.no_grad():
        test_tensor = torch.tensor(test_data, dtype=torch.float32).to(device)
        T = test_tensor.shape[1]
        timestamps = torch.linspace(0, 1, T).unsqueeze(0).expand(test_tensor.shape[0], -1).to(device)
        
        anomaly_scores = model.anomaly_score(test_tensor, timestamps)
        if isinstance(anomaly_scores, torch.Tensor):
            anomaly_scores = anomaly_scores.cpu().numpy()
    
    threshold = np.percentile(anomaly_scores, 95)
    predictions = (anomaly_scores > threshold).astype(int)
    
    precision = precision_score(test_labels, predictions, average='binary', zero_division=0)
    recall = recall_score(test_labels, predictions, average='binary', zero_division=0)
    f1 = f1_score(test_labels, predictions, average='binary', zero_division=0)
    
    try:
        auc = roc_auc_score(test_labels, anomaly_scores)
    except:
        auc = 0.5
    
    accuracy = np.mean(predictions == test_labels)
    
    print(f"임계값: {threshold:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC: {auc:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    
    cm = confusion_matrix(test_labels, predictions)
    print(f"\n혼동 행렬:")
    print(cm)
    
    return {
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'accuracy': accuracy,
        'confusion_matrix': cm,
        'anomaly_scores': anomaly_scores,
        'predictions': predictions
    }


In [36]:
"""
class Params:
    def __init__(self, dl, stride, tst_stride, train_method='train_all_entities', clamp_max=None, clamp_min=None, entity_id=None, downsample=None):
        self.dl = dl
        self.stride = stride
        self.tst_stride = tst_stride
        self.train_method = train_method
        self.clamp_max = clamp_max
        self.clamp_min = clamp_min
        self.entity_id = entity_id
        self.downsample = downsample
"""

"\nclass Params:\n    def __init__(self, dl, stride, tst_stride, train_method='train_all_entities', clamp_max=None, clamp_min=None, entity_id=None, downsample=None):\n        self.dl = dl\n        self.stride = stride\n        self.tst_stride = tst_stride\n        self.train_method = train_method\n        self.clamp_max = clamp_max\n        self.clamp_min = clamp_min\n        self.entity_id = entity_id\n        self.downsample = downsample\n"

In [37]:
def train_and_evaluate_robust_spike_detector(model, data_dict, num_epochs=200):
    """Robust Gradient Spike Detector를 위한 안전한 학습 및 평가 함수"""
    
    train_loader = data_dict['train_loader']
    x_tst = data_dict['x_tst']
    lab_tst = data_dict['lab_tst']
    
    device = next(model.parameters()).device
    
    print(f"=== Robust Gradient Spike Detector 학습 시작 (총 {num_epochs} 에포크) ===")
    
    # Gradient-aware 가중치 초기화
    sample_batch = next(iter(train_loader))
    if isinstance(sample_batch, (list, tuple)):
        sample_x = sample_batch[0]
    else:
        sample_x = sample_batch
    
    initializer = GradientAwareInitialization()
    initializer.init_weights_by_gradient_range(model, sample_x)
    
    # 옵티마이저 설정
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    model.to(device)
    model.train()
    
    history = []
    
    for epoch in range(num_epochs):
        epoch_losses = {
            'total_loss': 0,
            'recon_loss': 0,
            'kl_loss': 0,
            'grad_smoothness': 0
        }
        
        for batch_idx, batch_data in enumerate(train_loader):
            # 배치 데이터 처리
            if isinstance(batch_data, (list, tuple)):
                if len(batch_data) == 3:
                    batch_x, batch_times, batch_mask = batch_data
                    batch_x = batch_x.to(device)
                    batch_times = batch_times.to(device)
                    batch_mask = batch_mask.to(device)
                else:
                    batch_x, batch_times = batch_data
                    batch_x = batch_x.to(device)
                    batch_times = batch_times.to(device)
                    batch_mask = None
            else:
                batch_x = batch_data.to(device)
                batch_times = torch.linspace(0, 1, batch_x.size(1)).to(device)
                batch_mask = None
            
            # Gradient 초기화
            optimizer.zero_grad()
            
            try:
                # Forward pass
                outputs = model(batch_x, batch_times, batch_mask)
                losses = outputs['losses']
                
                # Backward pass
                losses['total_loss'].backward()
                
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                optimizer.step()
                
                # 손실 누적
                for key in epoch_losses.keys():
                    if key in losses:
                        epoch_losses[key] += losses[key].item()
                        
            except RuntimeError as e:
                print(f"Batch {batch_idx} failed: {e}")
                continue 
        
        n_batches = len(train_loader)
        avg_losses = {key: loss / n_batches for key, loss in epoch_losses.items()}
        
        scheduler.step()
        
        # 히스토리 저장
        history.append({
            'epoch': epoch + 1,
            **avg_losses,
            'lr': optimizer.param_groups[0]['lr']
        })
        
        # 주기적 로깅
        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{num_epochs}")
            print(f"  Total Loss: {avg_losses['total_loss']:.6f}")
            print(f"  Recon Loss: {avg_losses['recon_loss']:.6f}")
            print(f"  KL Loss: {avg_losses['kl_loss']:.6f}")
            print(f"  Grad Smoothness: {avg_losses['grad_smoothness']:.6f}")
    
    if lab_tst.ndim == 2:
        sequence_labels = (lab_tst.max(axis=1) > 0).astype(int)
    else:
        sequence_labels = lab_tst
    
    model.eval()
    
    print("=== 기본 평가 방법 ===")
    basic_metrics = simple_evaluate_robust_model(model, x_tst, sequence_labels, "Robust_Gradient_Spike_Detector")
    
    print("\n" + "="*60)
    comprehensive_metrics = comprehensive_evaluate_robust_model(model, x_tst, sequence_labels, "Robust_Gradient_Spike_Detector")
    
    return model, history, {'basic': basic_metrics, 'comprehensive': comprehensive_metrics}

def simple_evaluate_robust_model(model, x_tst, lab_tst, model_name):
    """Robust 모델을 위한 간단한 평가 함수"""
    device = next(model.parameters()).device
    
    if isinstance(x_tst, np.ndarray):
        x_tst = torch.tensor(x_tst, dtype=torch.float32)
    if isinstance(lab_tst, np.ndarray):
        lab_tst = torch.tensor(lab_tst, dtype=torch.float32)
    
    x_tst = x_tst.to(device)
    times = torch.linspace(0, 1, x_tst.size(1)).to(device)
    
    # 이상 점수 계산
    anomaly_scores = model.anomaly_score(x_tst, times)
    
    # 임계값 설정 (90th percentile)
    threshold = np.percentile(anomaly_scores, 90)
    predictions = (anomaly_scores > threshold).astype(int)
    
    # 메트릭 계산
    lab_tst_np = lab_tst.cpu().numpy() if isinstance(lab_tst, torch.Tensor) else lab_tst
    
    try:
        auc = roc_auc_score(lab_tst_np, anomaly_scores) if len(np.unique(lab_tst_np)) > 1 else 0.5
        f1 = f1_score(lab_tst_np, predictions, average='binary')
        precision = precision_score(lab_tst_np, predictions, average='binary', zero_division=0)
        recall = recall_score(lab_tst_np, predictions, average='binary', zero_division=0)
        accuracy = accuracy_score(lab_tst_np, predictions)
        
        print(f"\n{model_name} - Basic Evaluation:")
        print(f"AUC: {auc:.4f}")
        print(f"F1 Score: {f1:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Threshold: {threshold:.4f}")
        
        return {
            'auc': auc,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'accuracy': accuracy,
            'threshold': threshold,
            'anomaly_scores': anomaly_scores,
            'predictions': predictions
        }
        
    except Exception as e:
        print(f"평가 중 오류 발생: {e}")
        return {
            'auc': 0.0,
            'f1': 0.0,
            'precision': 0.0,
            'recall': 0.0,
            'accuracy': 0.0,
            'threshold': threshold,
            'error': str(e)
        }

def comprehensive_evaluate_robust_model(model, x_tst, lab_tst, model_name):
    """Robust 모델을 위한 포괄적 평가 함수"""
    device = next(model.parameters()).device
    
    if isinstance(x_tst, np.ndarray):
        x_tst = torch.tensor(x_tst, dtype=torch.float32)
    if isinstance(lab_tst, np.ndarray):
        lab_tst = torch.tensor(lab_tst, dtype=torch.float32)
    
    x_tst = x_tst.to(device)
    times = torch.linspace(0, 1, x_tst.size(1)).to(device)
    
    # 이상 점수 계산
    anomaly_scores = model.anomaly_score(x_tst, times)
    lab_tst_np = lab_tst.cpu().numpy() if isinstance(lab_tst, torch.Tensor) else lab_tst
    
    # 다양한 임계값으로 평가
    thresholds = np.percentile(anomaly_scores, [85, 90, 95, 99])
    
    results = {}
    best_f1 = 0
    best_threshold = thresholds[1]  # 기본값
    
    for i, threshold in enumerate(thresholds):
        predictions = (anomaly_scores > threshold).astype(int)
        
        try:
            f1 = f1_score(lab_tst_np, predictions, average='binary')
            precision = precision_score(lab_tst_np, predictions, average='binary', zero_division=0)
            recall = recall_score(lab_tst_np, predictions, average='binary', zero_division=0)
            
            percentile = [85, 90, 95, 99][i]
            results[f'threshold_{percentile}'] = {
                'threshold': threshold,
                'f1': f1,
                'precision': precision,
                'recall': recall
            }
            
            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold
                
        except Exception as e:
            print(f"임계값 {threshold:.4f}에서 평가 오류: {e}")
    
    # 최적 임계값으로 최종 평가
    final_predictions = (anomaly_scores > best_threshold).astype(int)
    
    try:
        final_metrics = {
            'auc': roc_auc_score(lab_tst_np, anomaly_scores) if len(np.unique(lab_tst_np)) > 1 else 0.5,
            'f1': f1_score(lab_tst_np, final_predictions, average='binary'),
            'precision': precision_score(lab_tst_np, final_predictions, average='binary', zero_division=0),
            'recall': recall_score(lab_tst_np, final_predictions, average='binary', zero_division=0),
            'accuracy': accuracy_score(lab_tst_np, final_predictions),
            'best_threshold': best_threshold
        }
        
        print(f"\n{model_name} - Comprehensive Evaluation:")
        print(f"Best AUC: {final_metrics['auc']:.4f}")
        print(f"Best F1 Score: {final_metrics['f1']:.4f}")
        print(f"Best Precision: {final_metrics['precision']:.4f}")
        print(f"Best Recall: {final_metrics['recall']:.4f}")
        print(f"Best Accuracy: {final_metrics['accuracy']:.4f}")
        print(f"Best Threshold: {final_metrics['best_threshold']:.4f}")
        
        # 임계값별 결과 출력
        print("\nThreshold Analysis:")
        for key, value in results.items():
            print(f"{key}: F1={value['f1']:.4f}, P={value['precision']:.4f}, R={value['recall']:.4f}")
        
        return {
            **final_metrics,
            'threshold_analysis': results,
            'anomaly_scores': anomaly_scores,
            'predictions': final_predictions
        }
        
    except Exception as e:
        print(f"최종 평가 중 오류 발생: {e}")
        return {
            'error': str(e),
            'threshold_analysis': results
        }

In [ ]:
def main():
    # 데이터 로드
    with open('./datasets/SMD/ServerMachineDataset/SMD.pk', 'rb') as f:
        data = pickle.load(f)
    
    entity_idx = 0
    x_trn = data['x_trn'][entity_idx]
    x_tst = data['x_tst'][entity_idx]
    lab_tst = data['lab_tst'][entity_idx]
    
    # 윈도우 레벨 라벨로 변환
    if lab_tst.ndim == 2:
        lab_tst = np.max(lab_tst, axis=1)
    
    data_dict = prepare_smd_data(x_trn, x_tst, lab_tst)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    
    # 모델 초기화
    input_dim = x_trn.shape[-1]
    model = RobustGradientSpikeDetector(
        input_dim=input_dim,
        latent_dim=32,
        hidden_dim=64,
        time_emb_dim=8,
        beta=0.1
    ).to(device)
    
    print(f"Input dim: {input_dim}")
    
    # 학습 및 평가
    model, history, metrics = train_and_evaluate_robust_spike_detector(
        model, 
        data_dict, 
        num_epochs=50
    )

    # 결과 출력
    basic = metrics['basic']
    comprehensive = metrics['comprehensive']
    
    print(f"\n=== 성능 결과 ===")
    print(f"F1:        {basic['f1']:.4f} → {comprehensive['f1']:.4f}")
    print(f"Precision: {basic['precision']:.4f} → {comprehensive['precision']:.4f}")
    print(f"Recall:    {basic['recall']:.4f} → {comprehensive['recall']:.4f}")
    print(f"AUC:       {basic['auc']:.4f}")
    
    # 성능 등급
    best_f1 = max(basic['f1'], comprehensive['f1'])
    if best_f1 >= 0.8:
        grade = "Excellent"
    elif best_f1 >= 0.7:
        grade = "Good"
    elif best_f1 >= 0.6:
        grade = "Fair"
    else:
        grade = "Under 0.5"
    
    print(f"Grade: {grade} (F1: {best_f1:.4f})")
    
    return model, history, metrics

def train_and_evaluate_robust_spike_detector(model, data_dict, num_epochs=50):
    
    train_loader = data_dict['train_loader']
    x_tst = data_dict['x_tst']
    lab_tst = data_dict['lab_tst']
    
    device = next(model.parameters()).device
    
    # 초기화
    sample_batch = next(iter(train_loader))
    if isinstance(sample_batch, (list, tuple)):
        sample_x = sample_batch[0]
    else:
        sample_x = sample_batch
    
    initializer = GradientAwareInitialization()
    initializer.init_weights_by_gradient_range(model, sample_x)
    
    # 옵티마이저
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    model.train()
    print(f"Training for {num_epochs} epochs...")
    
    for epoch in range(num_epochs):
        total_loss = 0
        n_batches = 0
        
        for batch_data in train_loader:
            # 배치 처리
            if isinstance(batch_data, (list, tuple)):
                if len(batch_data) == 3:
                    batch_x, batch_times, batch_mask = batch_data
                    batch_x = batch_x.to(device)
                    batch_times = batch_times.to(device)
                    batch_mask = batch_mask.to(device)
                else:
                    batch_x, batch_times = batch_data
                    batch_x = batch_x.to(device)
                    batch_times = batch_times.to(device)
                    batch_mask = None
            else:
                batch_x = batch_data.to(device)
                batch_times = torch.linspace(0, 1, batch_x.size(1)).to(device)
                batch_mask = None
            
            optimizer.zero_grad()
            
            try:
                outputs = model(batch_x, batch_times, batch_mask)
                loss = outputs['losses']['total_loss']
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                
                total_loss += loss.item()
                n_batches += 1
                        
            except RuntimeError:
                continue
        
        scheduler.step()
        
        if epoch % (num_epochs // 5) == 0:  # 5번만 출력
            avg_loss = total_loss / max(n_batches, 1)
            print(f"Epoch {epoch}/{num_epochs}: Loss = {avg_loss:.6f}")
    
    print("Training completed.")
    
    # 평가
    if lab_tst.ndim == 2:
        sequence_labels = (lab_tst.max(axis=1) > 0).astype(int)
    else:
        sequence_labels = lab_tst
    
    model.eval()
    
    basic_metrics = simple_evaluate_robust_model(model, x_tst, sequence_labels, "Basic")
    comprehensive_metrics = comprehensive_evaluate_robust_model(model, x_tst, sequence_labels, "Comprehensive")
    
    return model, [], {'basic': basic_metrics, 'comprehensive': comprehensive_metrics}

if __name__ == "__main__":
    model, history, metrics = main() 

### Dataset2 (TrimSyn)

In [39]:
import torch.optim as optim
import torch.nn as nn
import numpy as np
import pickle as pk
from torch.utils.data import DataLoader, TensorDataset
from torchdiffeq import odeint_adjoint as odeint

from sklearn.metrics import f1_score
import torch.nn.functional as F

In [40]:
# trimSyn 데이터 로드
trimSyn_path = './datasets/trimSyn/trimSyn.pk'
with open(trimSyn_path, 'rb') as pkf:
    data = pk.load(pkf)

x_trn_all = data['x_trn'] 
x_tst_all = data['x_tst']  
lab_tst_all = data['lab_tst']

print(f"트레이닝 데이터 수: {len(x_trn_all)}")
print(f"첫 번째 트레이닝 데이터 형태: {x_trn_all[0].shape}")
print(f"테스트 데이터 수: {len(x_tst_all)}")
for i, x_tst in enumerate(x_tst_all):
    print(f"테스트 섹션 {i} 형태: {x_tst.shape}, 라벨 형태: {lab_tst_all[i].shape}")

x_trn_concat = np.stack(x_trn_all, axis=0) 
x_trn_tensor = torch.tensor(x_trn_concat, dtype=torch.float32)

T = x_trn_tensor.shape[1]
time_steps = torch.linspace(0, 1, T) 

input_dim = x_trn_tensor.shape[2]
train_dataset = TensorDataset(x_trn_tensor, time_steps.repeat(x_trn_tensor.shape[0], 1)) 
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

트레이닝 데이터 수: 5
첫 번째 트레이닝 데이터 형태: (10000, 30)
테스트 데이터 수: 5
테스트 섹션 0 형태: (1720, 30), 라벨 형태: (1720,)
테스트 섹션 1 형태: (950, 30), 라벨 형태: (950,)
테스트 섹션 2 형태: (3070, 30), 라벨 형태: (3070,)
테스트 섹션 3 형태: (650, 30), 라벨 형태: (650,)
테스트 섹션 4 형태: (1290, 30), 라벨 형태: (1290,)


In [ ]:
def prepare_trimSyn_data(x_trn_all, x_tst_all, lab_tst_all, batch_size=64):
    """trimSyn 데이터를 위한 전처리 함수"""
    
    x_trn_concat = np.stack(x_trn_all, axis=0) 
    x_trn_tensor = torch.tensor(x_trn_concat, dtype=torch.float32)
    
    T = x_trn_tensor.shape[1]
    time_steps = torch.linspace(0, 1, T) 
    
    train_dataset = TensorDataset(x_trn_tensor, time_steps.repeat(x_trn_tensor.shape[0], 1)) 
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    return {
        'train_loader': train_loader,
        'x_tst': x_tst_all,
        'lab_tst': lab_tst_all
    }

def evaluate_trimSyn_sections(model, x_tst_all, lab_tst_all, model_name="TrimSyn"):
    """trimSyn의 여러 섹션을 평가하는 함수 (강화된 오류 처리)"""
    
    # 모델의 디바이스 확인 (함수 시작 시 한 번만)
    model_device = next(model.parameters()).device
    
    all_results = []
    
    for section_idx, (x_tst, lab_tst) in enumerate(zip(x_tst_all, lab_tst_all)):
        
        print(f"\n=== Section {section_idx} 분석 ===")
        print(f"데이터 형태: {x_tst.shape}")
        print(f"라벨 형태: {lab_tst.shape}")
        
        # 라벨 전처리 - 시퀀스 레벨로 변환
        if lab_tst.ndim == 2:
            # 2D 라벨: 각 시퀀스에서 하나라도 이상이 있으면 이상으로 분류
            sequence_labels = (lab_tst.max(axis=1) > 0).astype(int)
        elif lab_tst.ndim == 1 and len(lab_tst) == x_tst.shape[0]:
            # 1D 라벨이지만 시퀀스 개수와 일치
            sequence_labels = lab_tst.astype(int)
        else:
            # 다른 경우: 전체 시퀀스에 대한 단일 라벨로 처리
            if len(lab_tst.flatten()) == 1:
                sequence_labels = np.array([lab_tst.flatten()[0]] * x_tst.shape[0])
            else:
                sequence_labels = (lab_tst.max() > 0).astype(int)
                sequence_labels = np.array([sequence_labels] * x_tst.shape[0])
        
        print(f"처리된 라벨 형태: {sequence_labels.shape}")
        print(f"라벨 값 분포: {np.bincount(sequence_labels)}")
        print(f"이상치 비율: {np.mean(sequence_labels):.4f}")
        
        # 라벨 다양성 검사 (완전히 건너뛰지 않고 경고만)
        unique_labels = np.unique(sequence_labels)
        if len(unique_labels) < 2:
            print(f"⚠️  Section {section_idx}: 라벨 다양성 부족 (모든 라벨이 {unique_labels[0]})")
            print("   → 참고용으로만 평가 진행")
            force_evaluation = True
        else:
            force_evaluation = False
            
        # 텐서 생성 및 디바이스로 이동
        try:
            x_tst_tensor = torch.tensor(x_tst, dtype=torch.float32).to(model_device)
            if x_tst_tensor.dim() == 2:
                x_tst_tensor = x_tst_tensor.unsqueeze(0)
            
            T = x_tst_tensor.shape[1]
            times = torch.linspace(0, 1, T).unsqueeze(0).expand(x_tst_tensor.shape[0], -1).to(model_device)
            
            print(f"텐서 형태: {x_tst_tensor.shape}")
            print(f"타임스탬프 형태: {times.shape}")
            
        except Exception as e:
            print(f"❌ Section {section_idx}: 텐서 변환 실패 - {e}")
            continue
        
        try:
            # 이상 점수 계산
            with torch.no_grad():
                scores = model.anomaly_score(x_tst_tensor, times)
                if isinstance(scores, torch.Tensor):
                    scores = scores.cpu().numpy()
            
            print(f"원본 점수 형태: {scores.shape}")
            
            # 점수를 시퀀스 개수에 맞게 조정
            if scores.ndim > 1:
                # 여러 차원인 경우 적절히 집계
                if len(scores) == len(sequence_labels):
                    # 이미 시퀀스 개수와 일치
                    final_scores = scores.flatten() if scores.ndim > 1 else scores
                elif scores.shape[0] == 1 and len(sequence_labels) > 1:
                    # 단일 배치의 경우 복제
                    final_scores = np.repeat(scores.mean(), len(sequence_labels))
                else:
                    # 평균 또는 최대값으로 집계
                    final_scores = np.array([scores.mean()] * len(sequence_labels))
            else:
                final_scores = scores
            
            # 크기 매칭 확인 및 조정
            if len(final_scores) != len(sequence_labels):
                print(f"⚠️  점수({len(final_scores)})와 라벨({len(sequence_labels)}) 크기 불일치")
                min_len = min(len(final_scores), len(sequence_labels))
                final_scores = final_scores[:min_len]
                sequence_labels = sequence_labels[:min_len]
                print(f"   → 크기를 {min_len}으로 조정")
            
            print(f"최종 점수 형태: {final_scores.shape}")
            print(f"점수 범위: [{final_scores.min():.4f}, {final_scores.max():.4f}]")
            
            # AUC 계산 (라벨 다양성이 있는 경우만)
            if not force_evaluation:
                try:
                    auc = roc_auc_score(sequence_labels, final_scores)
                except Exception as auc_error:
                    print(f"⚠️  AUC 계산 실패: {auc_error}")
                    auc = 0.5
            else:
                auc = 0.5  # 기본값
                
            # 임계값 범위 설정
            thresholds = np.percentile(final_scores, np.linspace(50, 99, 20))
            
            best_f1 = 0
            best_metrics = None
            
            for threshold in thresholds:
                predictions = (final_scores > threshold).astype(int)
                
                # 성능 지표 계산
                try:
                    if not force_evaluation:
                        f1 = f1_score(sequence_labels, predictions, zero_division=0)
                        precision = precision_score(sequence_labels, predictions, zero_division=0)
                        recall = recall_score(sequence_labels, predictions, zero_division=0)
                    else:
                        # 라벨 다양성이 없는 경우 간단한 정확도만
                        f1 = accuracy_score(sequence_labels, predictions)
                        precision = f1
                        recall = f1
                        
                    accuracy = accuracy_score(sequence_labels, predictions)
                    
                    if f1 > best_f1:
                        best_f1 = f1
                        best_metrics = {
                            'section': section_idx,
                            'f1': f1,
                            'precision': precision,
                            'recall': recall,
                            'accuracy': accuracy,
                            'auc': auc,
                            'threshold': threshold,
                            'force_evaluation': force_evaluation,
                            'anomaly_ratio': np.mean(sequence_labels)
                        }
                except Exception as metric_error:
                    print(f"⚠️  지표 계산 오류 (임계값 {threshold:.4f}): {metric_error}")
                    continue
            
            if best_metrics:
                all_results.append(best_metrics)
                status = "참고용" if force_evaluation else "정상"
                print(f"✅ Section {section_idx} ({status}): F1={best_metrics['f1']:.4f}, AUC={best_metrics['auc']:.4f}")
            else:
                print(f"❌ Section {section_idx}: 유효한 메트릭을 계산할 수 없음")
                
        except Exception as e:
            print(f"❌ Section {section_idx} 전체 오류: {e}")
            continue
    
    # 결과 요약 및 분석
    if all_results:
        # 정상 평가와 참고용 평가 분리
        normal_results = [r for r in all_results if not r.get('force_evaluation', False)]
        reference_results = [r for r in all_results if r.get('force_evaluation', False)]
        
        print(f"\n{'='*60}")
        print(f"{model_name} 평가 결과 요약")
        print(f"{'='*60}")
        print(f"전체 섹션 수: {len(x_tst_all)}")
        print(f"정상 평가된 섹션: {len(normal_results)}")
        print(f"참고용 평가된 섹션: {len(reference_results)}")
        print(f"평가 실패 섹션: {len(x_tst_all) - len(all_results)}")
        
        if normal_results:
            # 정상 결과만으로 평균 계산
            avg_metrics = {
                'f1': np.mean([r['f1'] for r in normal_results]),
                'precision': np.mean([r['precision'] for r in normal_results]),
                'recall': np.mean([r['recall'] for r in normal_results]),
                'accuracy': np.mean([r['accuracy'] for r in normal_results]),
                'auc': np.mean([r['auc'] for r in normal_results])
            }
            
            print(f"\n=== 정상 섹션 평균 성능 ===")
            print(f"F1:        {avg_metrics['f1']:.4f}")
            print(f"Precision: {avg_metrics['precision']:.4f}")
            print(f"Recall:    {avg_metrics['recall']:.4f}")
            print(f"Accuracy:  {avg_metrics['accuracy']:.4f}")
            print(f"AUC:       {avg_metrics['auc']:.4f}")
            
        elif reference_results:
            # 참고용 결과라도 있으면 표시
            avg_metrics = {
                'f1': np.mean([r['f1'] for r in reference_results]),
                'precision': np.mean([r['precision'] for r in reference_results]),
                'recall': np.mean([r['recall'] for r in reference_results]),
                'accuracy': np.mean([r['accuracy'] for r in reference_results]),
                'auc': np.mean([r['auc'] for r in reference_results])
            }
            
            print(f"\n=== 참고용 섹션 평균 (라벨 다양성 부족) ===")
            print(f"F1/Accuracy: {avg_metrics['f1']:.4f}")
            print(f"AUC:         {avg_metrics['auc']:.4f}")
            print("⚠️  주의: 이 결과는 모든 라벨이 동일하여 참고용입니다.")
            
        else:
            avg_metrics = None
            
        # 섹션별 세부 정보
        if len(all_results) > 0:
            print(f"\n=== 섹션별 세부 결과 ===")
            for r in all_results:
                status = "참고용" if r.get('force_evaluation', False) else "정상"
                print(f"Section {r['section']} ({status}): "
                      f"F1={r['f1']:.4f}, AUC={r['auc']:.4f}, "
                      f"이상비율={r.get('anomaly_ratio', 0):.4f}")
        
        return avg_metrics
    else:
        print(f"\n❌ {model_name}: 평가 가능한 섹션이 없습니다")
        print("   → 모든 섹션에서 오류가 발생했습니다.")
        print("   → 데이터 형태나 모델 설정을 확인해주세요.")
        return None

def improved_train_and_evaluate_trimSyn(model, data_dict, num_epochs=50):
    """trimSyn을 위한 개선된 학습 및 평가 함수"""
    
    train_loader = data_dict['train_loader']
    x_tst_all = data_dict['x_tst']
    lab_tst_all = data_dict['lab_tst']
    
    device = next(model.parameters()).device
    
    sample_batch = next(iter(train_loader))
    if isinstance(sample_batch, (list, tuple)):
        sample_x = sample_batch[0]
    else:
        sample_x = sample_batch
    
    initializer = GradientAwareInitialization()
    initializer.init_weights_by_gradient_range(model, sample_x)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4, betas=(0.9, 0.999))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    
    model.train()
    print(f"Training for {num_epochs} epochs...")
    
    best_loss = float('inf')
    patience = 10
    patience_counter = 0
    
    for epoch in range(num_epochs):
        total_loss = 0
        recon_loss_sum = 0
        kl_loss_sum = 0
        grad_loss_sum = 0
        n_batches = 0
        
        for batch_data in train_loader:
            if isinstance(batch_data, (list, tuple)):
                if len(batch_data) == 3:
                    batch_x, batch_times, batch_mask = batch_data
                    batch_x = batch_x.to(device)
                    batch_times = batch_times.to(device)
                    batch_mask = batch_mask.to(device)
                else:
                    batch_x, batch_times = batch_data
                    batch_x = batch_x.to(device)
                    batch_times = batch_times.to(device)
                    batch_mask = None
            else:
                batch_x = batch_data.to(device)
                batch_times = torch.linspace(0, 1, batch_x.size(1)).to(device)
                batch_mask = None
            
            optimizer.zero_grad()
            
            try:
                outputs = model(batch_x, batch_times, batch_mask)
                losses = outputs['losses']
                
                beta_schedule = min(1.0, epoch / 20.0)
                
                total_batch_loss = (losses['recon_loss'] + 
                                  beta_schedule * losses['kl_loss'] + 
                                  0.01 * losses['grad_smoothness'])
                
                total_batch_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                optimizer.step()
                
                total_loss += total_batch_loss.item()
                recon_loss_sum += losses['recon_loss'].item()
                kl_loss_sum += losses['kl_loss'].item()
                grad_loss_sum += losses['grad_smoothness'].item()
                n_batches += 1
                        
            except RuntimeError:
                continue
        
        scheduler.step()
        
        avg_loss = total_loss / max(n_batches, 1)
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= patience and epoch > 20:
            print(f"Early stopping at epoch {epoch}")
            break
        
        if epoch % max(1, num_epochs // 10) == 0:
            print(f"Epoch {epoch}/{num_epochs}: Loss = {avg_loss:.6f} "
                  f"(R: {recon_loss_sum/n_batches:.4f}, "
                  f"KL: {kl_loss_sum/n_batches:.4f}, "
                  f"G: {grad_loss_sum/n_batches:.4f})")
    
    print("Training completed.")
    
    model.eval()
    
    # trimSyn 섹션별 평가
    metrics = evaluate_trimSyn_sections(model, x_tst_all, lab_tst_all, "Enhanced_TrimSyn")
    
    return model, [], metrics

def main():
    # trimSyn 데이터 로드
    trimSyn_path = './datasets/trimSyn/trimSyn.pk'
    with open(trimSyn_path, 'rb') as pkf:
        data = pickle.load(pkf)
    
    x_trn_all = data['x_trn'] 
    x_tst_all = data['x_tst']  
    lab_tst_all = data['lab_tst']
    
    print(f"트레이닝 데이터 수: {len(x_trn_all)}")
    print(f"첫 번째 트레이닝 데이터 형태: {x_trn_all[0].shape}")
    print(f"테스트 데이터 수: {len(x_tst_all)}")
    
    data_dict = prepare_trimSyn_data(x_trn_all, x_tst_all, lab_tst_all)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    
    input_dim = x_trn_all[0].shape[-1]
    
    model = RobustGradientSpikeDetector(
        input_dim=input_dim,
        latent_dim=48,
        hidden_dim=96,
        time_emb_dim=12,
        beta=0.05
    ).to(device)
    
    print(f"Input dim: {input_dim}")
    
    
    model, history, metrics = improved_train_and_evaluate_trimSyn(
        model, 
        data_dict, 
        num_epochs= 1
    )

    if metrics:
        best_f1 = metrics['f1']
        if best_f1 >= 0.8:
            grade = "Excellent"
        elif best_f1 >= 0.7:
            grade = "Good"
        elif best_f1 >= 0.6:
            grade = "Fair"
        else:
            grade = "Needs Improvement"
        
        print(f"\nGrade: {grade} (F1: {best_f1:.4f})")
    
    return model, history, metrics

In [ ]:
if __name__ == "__main__":
    model, history, metrics = main() 

트레이닝 데이터 수: 5
첫 번째 트레이닝 데이터 형태: (10000, 30)
테스트 데이터 수: 5
Device: cuda
Input dim: 30
Initialized weights with gradient_scale: 0.961936
Training for 50 epochs...
Epoch 0/50: Loss = 0.289833 (R: 0.2898, KL: 3.8504, G: 0.0001)


### Dataset 3 (SWaT)

In [49]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from datasets.SWaT.preprocess_SWaT import SWaT_Dataset
import pickle as pk

In [51]:
"""
trn = pd.read_csv('./datasets/SWaT/SWaT_Dataset_Normal_v1.csv')
tst = pd.read_csv('./datasets/SWaT/SWaT_Dataset_Attack_v0.csv', sep = ";")

channels = trn.columns[1:-1]
x_trn = trn[trn.columns[1:-1]].to_numpy()
x_tst = tst[tst.columns[1:-1]].to_numpy()
lab_tst = tst['Normal/Attack'].to_numpy()
lab_tst[lab_tst == 'Normal'] = 0
lab_tst[lab_tst == 'Attack'] = 1
lab_tst[lab_tst == 'A ttack'] = 1
lab_tst = np.array(lab_tst, dtype = int)

with open('./datasets/SWaT/SWaT.pk', 'wb') as file:
    pk.dump({'channels': channels, 'x_trn': x_trn, 'x_tst': x_tst, 'lab_tst': lab_tst}, file)

print('done')

"""

'\ntrn = pd.read_csv(\'./datasets/SWaT/SWaT_Dataset_Normal_v1.csv\')\ntst = pd.read_csv(\'./datasets/SWaT/SWaT_Dataset_Attack_v0.csv\', sep = ";")\n\nchannels = trn.columns[1:-1]\nx_trn = trn[trn.columns[1:-1]].to_numpy()\nx_tst = tst[tst.columns[1:-1]].to_numpy()\nlab_tst = tst[\'Normal/Attack\'].to_numpy()\nlab_tst[lab_tst == \'Normal\'] = 0\nlab_tst[lab_tst == \'Attack\'] = 1\nlab_tst[lab_tst == \'A ttack\'] = 1\nlab_tst = np.array(lab_tst, dtype = int)\n\nwith open(\'./datasets/SWaT/SWaT.pk\', \'wb\') as file:\n    pk.dump({\'channels\': channels, \'x_trn\': x_trn, \'x_tst\': x_tst, \'lab_tst\': lab_tst}, file)\n\nprint(\'done\')\n\n'

In [ ]:
dataset_path = './datasets/SWaT/SWaT.pk'

swat_dataset = SWaT_Dataset(dataset_path)

class Params:
    def __init__(self):
        self.dl = 100  # 데이터 길이
        self.stride = 50  # 스트라이드
        self.tst_stride = 'no_rep'  # 테스트 스트라이드
        
        # utils/preprocess.py에서 사용하는 추가 속성들
        self.clamp_max = None  # 최대값 클램핑
        self.clamp_min = None  # 최소값 클램핑
        self.downsample = None  # 다운샘플링
        self.entities = None  # 엔티티 수
        
params = Params()
processed_data = swat_dataset.preprocess(params)

In [57]:
x_trn_all = processed_data['x_trn'] 
x_tst_all = processed_data['x_tst']  
lab_tst_all = processed_data['lab_tst']

x_trn_concat = np.stack(x_trn_all, axis=0) 
x_trn_tensor = torch.tensor(x_trn_concat, dtype=torch.float32)

T = x_trn_tensor.shape[1]
time_steps = torch.linspace(0, 1, T) 

input_dim = x_trn_tensor.shape[2]
train_dataset = TensorDataset(x_trn_tensor, time_steps.repeat(x_trn_tensor.shape[0], 1)) 
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [27]:
def evaluate_SWaT_sections(model, x_tst_all, lab_tst_all, model_name="SWaT"):
    model_device = next(model.parameters()).device
    
    all_results = []
    
    for section_idx, (x_tst, lab_tst) in enumerate(zip(x_tst_all, lab_tst_all)):
        
        if lab_tst.ndim == 2:
            sequence_labels = (lab_tst.max(axis=1) > 0).astype(int)
        else:
            sequence_labels = lab_tst
        
        # 텐서 생성 및 디바이스로 이동
        x_tst_tensor = torch.tensor(x_tst, dtype=torch.float32).to(model_device)
        if x_tst_tensor.dim() == 2:
            x_tst_tensor = x_tst_tensor.unsqueeze(0)
        
        T = x_tst_tensor.shape[1]
        times = torch.linspace(0, 1, T).unsqueeze(0).expand(x_tst_tensor.shape[0], -1).to(model_device)
        
        try:
            scores = model.anomaly_score(x_tst_tensor, times)
            if isinstance(scores, torch.Tensor):
                scores = scores.cpu().numpy()
            
            # AUC 계산
            try:
                auc = roc_auc_score(sequence_labels, scores)
            except:
                auc = 0.5
            
            # 임계값 범위 설정
            thresholds = np.percentile(scores, np.linspace(50, 99, 50))
            
            best_f1 = 0
            best_metrics = None
            
            for threshold in thresholds:
                predictions = (scores > threshold).astype(int)
                
                f1 = f1_score(sequence_labels, predictions, zero_division=0)
                precision = precision_score(sequence_labels, predictions, zero_division=0)
                recall = recall_score(sequence_labels, predictions, zero_division=0)
                accuracy = accuracy_score(sequence_labels, predictions)
                
                if f1 > best_f1:
                    best_f1 = f1
                    best_metrics = {
                        'section': section_idx,
                        'f1': f1,
                        'precision': precision,
                        'recall': recall,
                        'accuracy': accuracy,
                        'auc': auc,
                        'threshold': threshold
                    }
            
            if best_metrics:
                all_results.append(best_metrics)
                print(f"Section {section_idx}: F1={best_metrics['f1']:.4f}, AUC={best_metrics['auc']:.4f}")
        
        except Exception as e:
            print(f"Section {section_idx} 오류: {e}")
            continue
    
    if all_results:
        avg_metrics = {
            'f1': np.mean([r['f1'] for r in all_results]),
            'precision': np.mean([r['precision'] for r in all_results]),
            'recall': np.mean([r['recall'] for r in all_results]),
            'accuracy': np.mean([r['accuracy'] for r in all_results]),
            'auc': np.mean([r['auc'] for r in all_results])
        }
        
        print(f"\n=== {model_name} 전체 평균 결과 ===")
        print(f"F1:        {avg_metrics['f1']:.4f}")
        print(f"Precision: {avg_metrics['precision']:.4f}")
        print(f"Recall:    {avg_metrics['recall']:.4f}")
        print(f"Accuracy:  {avg_metrics['accuracy']:.4f}")
        print(f"AUC:       {avg_metrics['auc']:.4f}")
        
        return avg_metrics
    else:
        print(f"❌ {model_name}: 평가 가능한 섹션이 없습니다")
        return None

print("✅ SWaT 섹션별 평가 함수가 정의되었습니다.")

✅ SWaT 섹션별 평가 함수가 정의되었습니다.


In [28]:
def prepare_SWaT_data(x_trn_all, x_tst_all, lab_tst_all, batch_size=64):
    """SWaT 데이터를 위한 전처리 함수"""
    
    x_trn_concat = np.stack(x_trn_all, axis=0) 
    x_trn_tensor = torch.tensor(x_trn_concat, dtype=torch.float32)
    
    T = x_trn_tensor.shape[1]
    time_steps = torch.linspace(0, 1, T) 
    
    train_dataset = TensorDataset(x_trn_tensor, time_steps.repeat(x_trn_tensor.shape[0], 1)) 
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    return {
        'train_loader': train_loader,
        'x_tst': x_tst_all,
        'lab_tst': lab_tst_all
    }

def main_SWaT():
    """SWaT 메인 실행 함수"""
    # SWaT 데이터 로드
    dataset_path = './datasets/SWaT/SWaT.pk'
    swat_dataset = SWaT_Dataset(dataset_path)
    
    class Params:
        def __init__(self):
            self.dl = 100  # 데이터 길이
            self.stride = 50  # 스트라이드
            self.tst_stride = 'no_rep'  # 테스트 스트라이드
            self.clamp_max = None  # 최대값 클램핑
            self.clamp_min = None  # 최소값 클램핑
            self.downsample = None  # 다운샘플링
            self.entities = None  # 엔티티 수
    
    params = Params()
    processed_data = swat_dataset.preprocess(params)
    
    x_trn_all = processed_data['x_trn'] 
    x_tst_all = processed_data['x_tst']  
    lab_tst_all = processed_data['lab_tst']
    
    print(f"트레이닝 데이터 수: {len(x_trn_all)}")
    print(f"첫 번째 트레이닝 데이터 형태: {x_trn_all[0].shape}")
    print(f"테스트 데이터 수: {len(x_tst_all)}")
    
    data_dict = prepare_SWaT_data(x_trn_all, x_tst_all, lab_tst_all)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    
    input_dim = x_trn_all[0].shape[-1]
    
    model = RobustGradientSpikeDetector(
        input_dim=input_dim,
        latent_dim=48,
        hidden_dim=96,
        time_emb_dim=12,
        beta=0.05
    ).to(device)
    
    print(f"Input dim: {input_dim}")
    
    model, history, metrics = improved_train_and_evaluate_SWaT(
        model, 
        data_dict, 
        num_epochs=50
    )

    if metrics:
        best_f1 = metrics['f1']
        if best_f1 >= 0.8:
            grade = "Excellent"
        elif best_f1 >= 0.7:
            grade = "Good"
        elif best_f1 >= 0.6:
            grade = "Fair"
        else:
            grade = "Needs Improvement"
        
        print(f"\nGrade: {grade} (F1: {best_f1:.4f})")
    
    return model, history, metrics

print("✅ SWaT 메인 함수가 정의되었습니다.")

✅ SWaT 메인 함수가 정의되었습니다.


In [29]:
if __name__ == "__main__":
    print("🚀 SWaT 데이터셋으로 모델 학습 및 평가를 시작합니다...")
    swat_model, swat_history, swat_metrics = main_SWaT()

🚀 SWaT 데이터셋으로 모델 학습 및 평가를 시작합니다...


NameError: name 'SWaT_Dataset' is not defined

In [30]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
import pandas as pd
from datetime import datetime, timedelta

def visualize_anomaly_detection_results(model, test_data, test_labels, section_name="Dataset", 
                                       feature_names=None, start_date=None, freq='H'):
    """
    이상 탐지 결과를 종합적으로 시각화하는 함수
    
    Args:
        model: 훈련된 모델
        test_data: 테스트 데이터 (numpy array)
        test_labels: 실제 라벨
        section_name: 섹션 이름
        feature_names: 피처 이름 리스트
        start_date: 시작 날짜 (시계열 x축용)
        freq: 데이터 주기 ('H', 'D', 'M' 등)
    """
    
    # 모델의 디바이스 확인
    model_device = next(model.parameters()).device
    model.eval()
    
    # 이상 점수 계산
    with torch.no_grad():
        test_tensor = torch.tensor(test_data, dtype=torch.float32).to(model_device)
        if test_tensor.dim() == 2:
            test_tensor = test_tensor.unsqueeze(0)
        
        T = test_tensor.shape[1]
        times = torch.linspace(0, 1, T).unsqueeze(0).expand(test_tensor.shape[0], -1).to(model_device)
        
        anomaly_scores = model.anomaly_score(test_tensor, times)
        if isinstance(anomaly_scores, torch.Tensor):
            anomaly_scores = anomaly_scores.cpu().numpy()
    
    # 임계값 계산
    threshold = np.percentile(anomaly_scores, 95)
    predictions = (anomaly_scores > threshold).astype(int)
    
    # 시간 인덱스 생성
    if start_date is not None:
        time_index = pd.date_range(start=start_date, periods=len(anomaly_scores), freq=freq)
    else:
        time_index = np.arange(len(anomaly_scores))
    
    # 피처 이름 설정
    if feature_names is None:
        n_features = test_data.shape[-1] if len(test_data.shape) > 1 else 1
        feature_names = [f'Feature_{i+1}' for i in range(min(n_features, 6))]
    
    # 서브플롯 생성
    fig, axes = plt.subplots(4, 2, figsize=(20, 16))
    fig.suptitle(f'{section_name} - Anomaly Detection Results', fontsize=16, fontweight='bold')
    
    # 1. 이상 점수 시계열
    ax1 = axes[0, 0]
    ax1.plot(time_index, anomaly_scores, color='blue', alpha=0.7, linewidth=1)
    ax1.axhline(y=threshold, color='red', linestyle='--', label=f'Threshold: {threshold:.3f}')
    ax1.fill_between(time_index, 0, anomaly_scores, where=(anomaly_scores > threshold), 
                     color='red', alpha=0.3, label='Anomalies')
    ax1.set_title('Anomaly Scores Over Time')
    ax1.set_ylabel('Anomaly Score')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. 이상 점수 분포
    ax2 = axes[0, 1]
    ax2.hist(anomaly_scores, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    ax2.axvline(x=threshold, color='red', linestyle='--', label=f'Threshold: {threshold:.3f}')
    ax2.set_title('Anomaly Score Distribution')
    ax2.set_xlabel('Anomaly Score')
    ax2.set_ylabel('Frequency')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. 실제 vs 예측 비교
    ax3 = axes[1, 0]
    ax3.plot(time_index, test_labels, color='green', label='True Labels', linewidth=2)
    ax3.plot(time_index, predictions, color='red', alpha=0.7, label='Predictions', linewidth=1)
    ax3.fill_between(time_index, 0, 1, where=(test_labels == 1), 
                     color='green', alpha=0.2, label='True Anomalies')
    ax3.fill_between(time_index, 0, 1, where=(predictions == 1), 
                     color='red', alpha=0.2, label='Predicted Anomalies')
    ax3.set_title('True vs Predicted Labels')
    ax3.set_ylabel('Label')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. 혼동 행렬
    ax4 = axes[1, 1]
    cm = confusion_matrix(test_labels, predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4)
    ax4.set_title('Confusion Matrix')
    ax4.set_xlabel('Predicted')
    ax4.set_ylabel('Actual')
    
    # 5. ROC 곡선
    ax5 = axes[2, 0]
    fpr, tpr, _ = roc_curve(test_labels, anomaly_scores)
    auc_score = roc_auc_score(test_labels, anomaly_scores)
    ax5.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {auc_score:.3f})')
    ax5.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random')
    ax5.set_title('ROC Curve')
    ax5.set_xlabel('False Positive Rate')
    ax5.set_ylabel('True Positive Rate')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # 6. Precision-Recall 곡선
    ax6 = axes[2, 1]
    precision, recall, _ = precision_recall_curve(test_labels, anomaly_scores)
    ax6.plot(recall, precision, color='purple')
    ax6.set_title('Precision-Recall Curve')
    ax6.set_xlabel('Recall')
    ax6.set_ylabel('Precision')
    ax6.grid(True, alpha=0.3)
    
    # 7. 일부 피처들의 시계열 데이터 (이상 구간 하이라이트)
    ax7 = axes[3, 0]
    if len(test_data.shape) > 1:
        n_features_to_show = min(3, test_data.shape[-1])
        for i in range(n_features_to_show):
            if test_data.ndim == 3:
                feature_data = test_data[0, :, i]  # 첫 번째 시퀀스
            else:
                feature_data = test_data[:, i]
            ax7.plot(time_index[:len(feature_data)], feature_data, 
                    label=feature_names[i], alpha=0.8)
        
        # 이상 구간 하이라이트
        anomaly_periods = []
        in_anomaly = False
        start_idx = None
        
        for i, is_anomaly in enumerate(predictions):
            if is_anomaly and not in_anomaly:
                in_anomaly = True
                start_idx = i
            elif not is_anomaly and in_anomaly:
                in_anomaly = False
                if start_idx is not None:
                    anomaly_periods.append((start_idx, i-1))
        
        if in_anomaly and start_idx is not None:
            anomaly_periods.append((start_idx, len(predictions)-1))
        
        for start, end in anomaly_periods:
            ax7.axvspan(time_index[start], time_index[end], alpha=0.2, color='red')
    
    ax7.set_title('Original Data with Anomaly Highlights')
    ax7.set_ylabel('Value')
    ax7.legend()
    ax7.grid(True, alpha=0.3)
    
    # 8. 성능 지표 표
    ax8 = axes[3, 1]
    ax8.axis('off')
    
    # 성능 지표 계산
    precision_val = precision_score(test_labels, predictions, zero_division=0)
    recall_val = recall_score(test_labels, predictions, zero_division=0)
    f1_val = f1_score(test_labels, predictions, zero_division=0)
    accuracy_val = accuracy_score(test_labels, predictions)
    
    metrics_text = f'''Performance Metrics:
    
Accuracy:  {accuracy_val:.4f}
Precision: {precision_val:.4f}
Recall:    {recall_val:.4f}
F1-Score:  {f1_val:.4f}
AUC:       {auc_score:.4f}

Threshold: {threshold:.4f}
Total Samples: {len(test_labels)}
True Positives: {cm[1,1]}
False Positives: {cm[0,1]}
True Negatives: {cm[0,0]}
False Negatives: {cm[1,0]}'''
    
    ax8.text(0.1, 0.5, metrics_text, fontsize=12, verticalalignment='center',
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    return {
        'anomaly_scores': anomaly_scores,
        'predictions': predictions,
        'threshold': threshold,
        'metrics': {
            'accuracy': accuracy_val,
            'precision': precision_val,
            'recall': recall_val,
            'f1': f1_val,
            'auc': auc_score
        }
    }

print("✅ 종합 시각화 함수가 정의되었습니다.")


✅ 종합 시각화 함수가 정의되었습니다.


In [31]:
def visualize_sections_comparison(all_results, dataset_name="Dataset"):
    """
    여러 섹션의 성능을 비교하는 시각화 함수
    
    Args:
        all_results: 섹션별 결과 리스트 (각 항목은 metrics dict 포함)
        dataset_name: 데이터셋 이름
    """
    
    if not all_results:
        print("비교할 결과가 없습니다.")
        return
    
    # 데이터 준비
    sections = [f"Section {r['section']}" for r in all_results]
    metrics = ['f1', 'precision', 'recall', 'accuracy', 'auc']
    
    # 서브플롯 생성
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'{dataset_name} - Sections Performance Comparison', fontsize=16, fontweight='bold')
    
    # 1. 모든 지표 비교 (막대 그래프)
    ax1 = axes[0, 0]
    x = np.arange(len(sections))
    width = 0.15
    
    for i, metric in enumerate(metrics):
        values = [r[metric] for r in all_results]
        ax1.bar(x + i*width, values, width, label=metric.upper(), alpha=0.8)
    
    ax1.set_title('All Metrics Comparison')
    ax1.set_xlabel('Sections')
    ax1.set_ylabel('Score')
    ax1.set_xticks(x + width * 2)
    ax1.set_xticklabels(sections, rotation=45)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. F1 점수 추세
    ax2 = axes[0, 1]
    f1_scores = [r['f1'] for r in all_results]
    ax2.plot(sections, f1_scores, marker='o', linewidth=2, markersize=8, color='blue')
    ax2.set_title('F1 Score Trend')
    ax2.set_ylabel('F1 Score')
    ax2.grid(True, alpha=0.3)
    plt.setp(ax2.get_xticklabels(), rotation=45)
    
    # 3. AUC 점수 추세
    ax3 = axes[0, 2]
    auc_scores = [r['auc'] for r in all_results]
    ax3.plot(sections, auc_scores, marker='s', linewidth=2, markersize=8, color='red')
    ax3.set_title('AUC Score Trend')
    ax3.set_ylabel('AUC Score')
    ax3.grid(True, alpha=0.3)
    plt.setp(ax3.get_xticklabels(), rotation=45)
    
    # 4. Precision vs Recall 산점도
    ax4 = axes[1, 0]
    precision_scores = [r['precision'] for r in all_results]
    recall_scores = [r['recall'] for r in all_results]
    scatter = ax4.scatter(recall_scores, precision_scores, s=100, alpha=0.7, c=f1_scores, cmap='viridis')
    ax4.set_title('Precision vs Recall')
    ax4.set_xlabel('Recall')
    ax4.set_ylabel('Precision')
    ax4.grid(True, alpha=0.3)
    
    # 각 점에 섹션 이름 표시
    for i, section in enumerate(sections):
        ax4.annotate(section, (recall_scores[i], precision_scores[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    # 컬러바 추가
    cbar = plt.colorbar(scatter, ax=ax4)
    cbar.set_label('F1 Score')
    
    # 5. 평균 성능 레이더 차트
    ax5 = axes[1, 1]
    
    # 전체 평균 계산
    avg_metrics = {}
    for metric in metrics:
        avg_metrics[metric] = np.mean([r[metric] for r in all_results])
    
    # 레이더 차트를 위한 각도 계산
    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]  # 원을 닫기 위해
    
    values = list(avg_metrics.values())
    values += values[:1]  # 원을 닫기 위해
    
    ax5 = plt.subplot(2, 3, 5, projection='polar')
    ax5.plot(angles, values, 'o-', linewidth=2, color='green')
    ax5.fill(angles, values, alpha=0.25, color='green')
    ax5.set_xticks(angles[:-1])
    ax5.set_xticklabels([m.upper() for m in metrics])
    ax5.set_ylim(0, 1)
    ax5.set_title('Average Performance Radar', y=1.08)
    ax5.grid(True)
    
    # 6. 성능 통계 요약 표
    ax6 = axes[1, 2]
    ax6.axis('off')
    
    # 통계 계산
    stats_text = f'''Performance Statistics:
    
Best F1: {max(f1_scores):.4f} (Section {all_results[np.argmax(f1_scores)]['section']})
Worst F1: {min(f1_scores):.4f} (Section {all_results[np.argmin(f1_scores)]['section']})
Avg F1: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}

Best AUC: {max(auc_scores):.4f} (Section {all_results[np.argmax(auc_scores)]['section']})
Worst AUC: {min(auc_scores):.4f} (Section {all_results[np.argmin(auc_scores)]['section']})
Avg AUC: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}

Total Sections: {len(all_results)}
Sections with F1 > 0.8: {sum(1 for f1 in f1_scores if f1 > 0.8)}
Sections with F1 > 0.7: {sum(1 for f1 in f1_scores if f1 > 0.7)}'''
    
    ax6.text(0.1, 0.5, stats_text, fontsize=11, verticalalignment='center',
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgreen", alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    return avg_metrics

def create_anomaly_timeline(anomaly_scores, predictions, test_labels, time_index=None, 
                           section_name="Timeline", figsize=(15, 8)):
    """
    이상 탐지 결과의 타임라인 시각화
    
    Args:
        anomaly_scores: 이상 점수 배열
        predictions: 예측 결과
        test_labels: 실제 라벨
        time_index: 시간 인덱스
        section_name: 섹션 이름
        figsize: 그래프 크기
    """
    
    if time_index is None:
        time_index = np.arange(len(anomaly_scores))
    
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)
    fig.suptitle(f'{section_name} - Anomaly Detection Timeline', fontsize=14, fontweight='bold')
    
    # 1. 이상 점수
    ax1 = axes[0]
    ax1.plot(time_index, anomaly_scores, color='blue', alpha=0.7, linewidth=1)
    threshold = np.percentile(anomaly_scores, 95)
    ax1.axhline(y=threshold, color='red', linestyle='--', alpha=0.8)
    ax1.fill_between(time_index, 0, anomaly_scores, where=(anomaly_scores > threshold), 
                     color='red', alpha=0.3, interpolate=True)
    ax1.set_ylabel('Anomaly Score')
    ax1.set_title('Anomaly Scores Over Time')
    ax1.grid(True, alpha=0.3)
    
    # 2. 예측 vs 실제
    ax2 = axes[1]
    ax2.plot(time_index, test_labels * 0.9, color='green', linewidth=3, alpha=0.8, label='True')
    ax2.plot(time_index, predictions * 1.1, color='red', linewidth=2, alpha=0.8, label='Predicted')
    ax2.fill_between(time_index, 0, 1.2, where=(test_labels == 1), 
                     color='green', alpha=0.2, interpolate=True)
    ax2.fill_between(time_index, 0, 1.2, where=(predictions == 1), 
                     color='red', alpha=0.2, interpolate=True)
    ax2.set_ylabel('Labels')
    ax2.set_title('True vs Predicted Labels')
    ax2.set_ylim(-0.1, 1.3)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. 오류 분석
    ax3 = axes[2]
    
    # False Positives와 False Negatives 계산
    fp = (predictions == 1) & (test_labels == 0)
    fn = (predictions == 0) & (test_labels == 1)
    tp = (predictions == 1) & (test_labels == 1)
    tn = (predictions == 0) & (test_labels == 0)
    
    # 각 유형별로 다른 색상으로 표시
    ax3.scatter(time_index[tp], np.ones(np.sum(tp)) * 1, color='green', s=20, alpha=0.8, label='True Positive')
    ax3.scatter(time_index[fp], np.ones(np.sum(fp)) * 0.5, color='red', s=20, alpha=0.8, label='False Positive')
    ax3.scatter(time_index[fn], np.ones(np.sum(fn)) * 0, color='orange', s=20, alpha=0.8, label='False Negative')
    
    ax3.set_ylabel('Error Type')
    ax3.set_xlabel('Time')
    ax3.set_title('Error Analysis')
    ax3.set_ylim(-0.2, 1.2)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 오류 통계 출력
    print(f"\n📊 {section_name} Error Analysis:")
    print(f"True Positives: {np.sum(tp)}")
    print(f"False Positives: {np.sum(fp)}")
    print(f"False Negatives: {np.sum(fn)}")
    print(f"True Negatives: {np.sum(tn)}")
    
    return {
        'tp': np.sum(tp),
        'fp': np.sum(fp),
        'fn': np.sum(fn),
        'tn': np.sum(tn)
    }

print("✅ 섹션 비교 및 타임라인 시각화 함수가 정의되었습니다.")


✅ 섹션 비교 및 타임라인 시각화 함수가 정의되었습니다.


In [32]:
def evaluate_with_visualization(evaluation_function, model, x_tst_all, lab_tst_all, 
                               model_name="Model", show_individual=True, show_comparison=True):
    """
    평가 함수에 시각화를 추가하는 래퍼 함수
    
    Args:
        evaluation_function: 기존 평가 함수 (evaluate_trimSyn_sections 또는 evaluate_SWaT_sections)
        model: 훈련된 모델
        x_tst_all: 테스트 데이터 리스트
        lab_tst_all: 테스트 라벨 리스트
        model_name: 모델 이름
        show_individual: 개별 섹션 시각화 여부
        show_comparison: 섹션 비교 시각화 여부
    """
    
    print(f"🔍 {model_name} 평가 및 시각화를 시작합니다...")
    
    # 기본 평가 수행 및 개별 결과 수집
    model_device = next(model.parameters()).device
    all_results = []
    
    for section_idx, (x_tst, lab_tst) in enumerate(zip(x_tst_all, lab_tst_all)):
        
        if lab_tst.ndim == 2:
            sequence_labels = (lab_tst.max(axis=1) > 0).astype(int)
        else:
            sequence_labels = lab_tst
        
        # 텐서 생성 및 디바이스로 이동
        x_tst_tensor = torch.tensor(x_tst, dtype=torch.float32).to(model_device)
        if x_tst_tensor.dim() == 2:
            x_tst_tensor = x_tst_tensor.unsqueeze(0)
        
        T = x_tst_tensor.shape[1]
        times = torch.linspace(0, 1, T).unsqueeze(0).expand(x_tst_tensor.shape[0], -1).to(model_device)
        
        try:
            scores = model.anomaly_score(x_tst_tensor, times)
            if isinstance(scores, torch.Tensor):
                scores = scores.cpu().numpy()
            
            # AUC 계산
            try:
                auc = roc_auc_score(sequence_labels, scores)
            except:
                auc = 0.5
            
            # 임계값 범위 설정
            thresholds = np.percentile(scores, np.linspace(50, 99, 50))
            
            best_f1 = 0
            best_metrics = None
            
            for threshold in thresholds:
                predictions = (scores > threshold).astype(int)
                
                f1 = f1_score(sequence_labels, predictions, zero_division=0)
                precision = precision_score(sequence_labels, predictions, zero_division=0)
                recall = recall_score(sequence_labels, predictions, zero_division=0)
                accuracy = accuracy_score(sequence_labels, predictions)
                
                if f1 > best_f1:
                    best_f1 = f1
                    best_metrics = {
                        'section': section_idx,
                        'f1': f1,
                        'precision': precision,
                        'recall': recall,
                        'accuracy': accuracy,
                        'auc': auc,
                        'threshold': threshold
                    }
            
            if best_metrics:
                all_results.append(best_metrics)
                print(f"Section {section_idx}: F1={best_metrics['f1']:.4f}, AUC={best_metrics['auc']:.4f}")
                
                # 개별 섹션 시각화
                if show_individual and section_idx < 3:  # 처음 3개 섹션만 시각화
                    print(f"\n📊 Section {section_idx} 상세 시각화:")
                    viz_result = visualize_anomaly_detection_results(
                        model, x_tst, sequence_labels, 
                        section_name=f"{model_name} Section {section_idx}",
                        start_date='2020-01-01',
                        freq='H'
                    )
        
        except Exception as e:
            print(f"Section {section_idx} 오류: {e}")
            continue
    
    # 전체 결과 계산
    if all_results:
        avg_metrics = {
            'f1': np.mean([r['f1'] for r in all_results]),
            'precision': np.mean([r['precision'] for r in all_results]),
            'recall': np.mean([r['recall'] for r in all_results]),
            'accuracy': np.mean([r['accuracy'] for r in all_results]),
            'auc': np.mean([r['auc'] for r in all_results])
        }
        
        print(f"\n=== {model_name} 전체 평균 결과 ===")
        print(f"F1:        {avg_metrics['f1']:.4f}")
        print(f"Precision: {avg_metrics['precision']:.4f}")
        print(f"Recall:    {avg_metrics['recall']:.4f}")
        print(f"Accuracy:  {avg_metrics['accuracy']:.4f}")
        print(f"AUC:       {avg_metrics['auc']:.4f}")
        
        # 섹션 비교 시각화
        if show_comparison and len(all_results) > 1:
            print(f"\n📈 {model_name} 섹션별 성능 비교:")
            comparison_result = visualize_sections_comparison(all_results, model_name)
        
        return avg_metrics
    else:
        print(f"❌ {model_name}: 평가 가능한 섹션이 없습니다")
        return None

# 기존 평가 함수들을 시각화 버전으로 래핑
def evaluate_trimSyn_sections_with_viz(model, x_tst_all, lab_tst_all, model_name="TrimSyn_Enhanced"):
    """TrimSyn 섹션별 평가 (시각화 포함)"""
    return evaluate_with_visualization(
        evaluate_trimSyn_sections, model, x_tst_all, lab_tst_all, model_name
    )

def evaluate_SWaT_sections_with_viz(model, x_tst_all, lab_tst_all, model_name="SWaT_Enhanced"):
    """SWaT 섹션별 평가 (시각화 포함)"""
    return evaluate_with_visualization(
        evaluate_SWaT_sections, model, x_tst_all, lab_tst_all, model_name
    )

print("✅ 시각화 통합 평가 함수가 정의되었습니다.")


✅ 시각화 통합 평가 함수가 정의되었습니다.


In [33]:
# TrimSyn 시각화 예제
# 모델이 학습된 후 다음 코드를 실행하여 시각화를 확인할 수 있습니다.

def run_trimSyn_with_visualization():
    """TrimSyn 데이터로 모델 학습 및 시각화 수행"""
    
    # TrimSyn 데이터 로드
    trimSyn_path = './datasets/trimSyn/trimSyn.pk'
    with open(trimSyn_path, 'rb') as pkf:
        data = pickle.load(pkf)
    
    x_trn_all = data['x_trn'] 
    x_tst_all = data['x_tst']  
    lab_tst_all = data['lab_tst']
    
    print(f"🎯 TrimSyn 시각화 데모 시작")
    print(f"트레이닝 데이터 수: {len(x_trn_all)}")
    print(f"테스트 데이터 수: {len(x_tst_all)}")
    
    data_dict = prepare_trimSyn_data(x_trn_all, x_tst_all, lab_tst_all)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    
    input_dim = x_trn_all[0].shape[-1]
    
    model = RobustGradientSpikeDetector(
        input_dim=input_dim,
        latent_dim=48,
        hidden_dim=96,
        time_emb_dim=12,
        beta=0.05
    ).to(device)
    
    print(f"Input dim: {input_dim}")
    
    # 모델 학습
    model, history, metrics = improved_train_and_evaluate_trimSyn(
        model, 
        data_dict, 
        num_epochs=30  # 시각화 데모를 위해 에포크 수 감소
    )
    
    # 시각화 포함 평가
    print(f"\n🎨 시각화를 포함한 상세 평가를 시작합니다...")
    viz_metrics = evaluate_trimSyn_sections_with_viz(
        model, x_tst_all, lab_tst_all, "TrimSyn_Visualization_Demo"
    )
    
    return model, history, viz_metrics

def run_single_section_visualization_demo():
    """단일 섹션 시각화 데모"""
    
    # 간단한 가상 데이터로 시각화 데모
    print("🎨 단일 섹션 시각화 데모")
    
    # 가상 데이터 생성
    np.random.seed(42)
    
    # 정상 데이터
    normal_data = np.random.normal(0, 1, (800, 10))
    normal_labels = np.zeros(800)
    
    # 이상 데이터 (몇 개 구간에 이상치 추가)
    anomaly_indices = [100, 200, 300, 500, 600]
    for idx in anomaly_indices:
        normal_data[idx:idx+20] += np.random.normal(3, 0.5, (20, 10))
        normal_labels[idx:idx+20] = 1
    
    # 시간 인덱스 생성
    time_index = pd.date_range('2023-01-01', periods=800, freq='H')
    
    # 가상 모델 결과 (실제로는 모델에서 계산)
    anomaly_scores = np.random.normal(0.3, 0.1, 800)
    for idx in anomaly_indices:
        anomaly_scores[idx:idx+20] += np.random.normal(0.4, 0.1, 20)
    
    anomaly_scores = np.clip(anomaly_scores, 0, 1)
    threshold = np.percentile(anomaly_scores, 95)
    predictions = (anomaly_scores > threshold).astype(int)
    
    # 타임라인 시각화
    timeline_stats = create_anomaly_timeline(
        anomaly_scores, predictions, normal_labels, time_index, 
        "Demo Timeline", figsize=(16, 10)
    )
    
    print("✅ 시각화 데모가 완료되었습니다!")
    return timeline_stats

print("✅ 시각화 데모 함수가 정의되었습니다.")


✅ 시각화 데모 함수가 정의되었습니다.
